# ACC-vmPFC nested

# splitted version

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.model_selection import StratifiedKFold, KFold, cross_val_score
from sklearn.metrics import roc_auc_score, make_scorer, r2_score

import statsmodels.formula.api as smf

warnings.filterwarnings("ignore")

# =========================================================
# paths
# =========================================================
INPUT_CSV = "/root/reveal_neural_feature_tables/reveal_level_neural_feature_table.csv"
OUTPUT_DIR = "/root/reveal_roi_state_space_results/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================================================
# load
# =========================================================
df = pd.read_csv(INPUT_CSV)
print("Loaded:", df.shape)

# 只保留完整 6-reveal game
df = df[df["game_complete_6"] == True].copy()
df = df.sort_values(["Subject", "Game", "reveal_idx"]).reset_index(drop=True)
print("After complete-game filter:", df.shape)

# =========================================================
# helper: pick feature columns by ROI
# 第一版仍然尽量稳：bin + full_mean + full_auc + channel_sd_full
# =========================================================
def pick_roi_features(columns, roi_name):
    feats = []
    for c in columns:
        if not c.startswith(f"{roi_name}_"):
            continue
        if (
            "_bin" in c
            or c.endswith("_full_mean")
            or c.endswith("_full_auc")
            or c.endswith("_channel_sd_full")
        ):
            feats.append(c)
    return feats

acc_cols = pick_roi_features(df.columns, "acc")
vmpfc_cols = pick_roi_features(df.columns, "vmpfc")

# 去掉全空列
acc_cols = [c for c in acc_cols if df[c].notna().sum() > 0]
vmpfc_cols = [c for c in vmpfc_cols if df[c].notna().sum() > 0]

print("ACC feature n =", len(acc_cols))
print("vmPFC feature n =", len(vmpfc_cols))

# =========================================================
# sanity check
# =========================================================
required_cols = ["Subject", "Game", "condition_label", "reveal_idx"]
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

# 一些后面会用到的行为/任务变量，可能有些不存在
candidate_targets = [
    "current_reward",
    "best_deck_snapshot",
    "value_gap_snapshot",
    "sampling_imbalance_after",
    "sampling_imbalance_before",
    "imbalance_change",
    "current_deck_was_least_sampled_before",
    "current_deck_is_best_snapshot",
]

available_targets = [c for c in candidate_targets if c in df.columns]
print("Available targets:", available_targets)

# =========================================================
# helper: preprocessing
# =========================================================
def preprocess_matrix(X):
    imputer = SimpleImputer(strategy="median")
    X_imp = imputer.fit_transform(X)

    scaler = StandardScaler()
    X_std = scaler.fit_transform(X_imp)
    return X_std, imputer, scaler

# =========================================================
# helper: subject-centered / subject-condition-centered
# mode:
#   "raw"
#   "subject_centered"
#   "subject_condition_centered"
# =========================================================
def center_features(df_in, feature_cols, mode="subject_centered"):
    X = df_in[feature_cols].copy()

    if mode == "raw":
        return X

    out = X.copy()

    if mode == "subject_centered":
        subj_means = df_in.groupby("Subject")[feature_cols].transform("mean")
        out = X - subj_means

    elif mode == "subject_condition_centered":
        # 每个被试在每个 condition 内中心化
        sc_means = df_in.groupby(["Subject", "condition_label"])[feature_cols].transform("mean")
        out = X - sc_means

    else:
        raise ValueError(f"Unknown center mode: {mode}")

    return out

# =========================================================
# helper: fit PCA for one ROI
# return:
#   df scores
#   loadings
#   explained variance
# =========================================================
def run_roi_pca(df_in, roi_name, feature_cols, center_mode="subject_centered", n_components=3):
    if len(feature_cols) == 0:
        raise ValueError(f"No features for ROI={roi_name}")

    X_centered = center_features(df_in, feature_cols, mode=center_mode)
    X_std, imputer, scaler = preprocess_matrix(X_centered.to_numpy(dtype=float))

    pca = PCA(n_components=n_components)
    X_pca = pca.fit_transform(X_std)

    score_df = df_in[["Subject", "Game", "condition_label", "reveal_idx"]].copy()
    for i in range(n_components):
        score_df[f"{roi_name}_PC{i+1}"] = X_pca[:, i]

    loadings = pd.DataFrame(
        pca.components_.T,
        index=feature_cols,
        columns=[f"{roi_name}_PC{i+1}" for i in range(n_components)]
    )

    explained = pd.Series(
        pca.explained_variance_ratio_,
        index=[f"{roi_name}_PC{i+1}" for i in range(n_components)],
        name="explained_variance_ratio"
    )

    return score_df, loadings, explained, pca, imputer, scaler

# =========================================================
# run ROI-specific PCA
# =========================================================
acc_scores, acc_loadings, acc_explained, acc_pca, _, _ = run_roi_pca(
    df, "acc", acc_cols, center_mode="subject_centered", n_components=3
)

vmpfc_scores, vmpfc_loadings, vmpfc_explained, vmpfc_pca, _, _ = run_roi_pca(
    df, "vmpfc", vmpfc_cols, center_mode="subject_centered", n_components=3
)

acc_loadings.to_csv(os.path.join(OUTPUT_DIR, "acc_pca_loadings.csv"))
vmpfc_loadings.to_csv(os.path.join(OUTPUT_DIR, "vmpfc_pca_loadings.csv"))
acc_explained.to_csv(os.path.join(OUTPUT_DIR, "acc_explained_variance.csv"), header=True)
vmpfc_explained.to_csv(os.path.join(OUTPUT_DIR, "vmpfc_explained_variance.csv"), header=True)

# merge back
df_state = df.copy()
for col in acc_scores.columns:
    if col not in ["Subject", "Game", "condition_label", "reveal_idx"]:
        df_state[col] = acc_scores[col].values
for col in vmpfc_scores.columns:
    if col not in ["Subject", "Game", "condition_label", "reveal_idx"]:
        df_state[col] = vmpfc_scores[col].values

# ROI divergence / coupling proxy
df_state["roi_pc1_diff"] = df_state["acc_PC1"] - df_state["vmpfc_PC1"]
df_state["roi_pc2_diff"] = df_state["acc_PC2"] - df_state["vmpfc_PC2"]
df_state["roi_pc3_diff"] = df_state["acc_PC3"] - df_state["vmpfc_PC3"]

# =========================================================
# plot subject-averaged ROI trajectories
# =========================================================
def plot_roi_trajectory(df_in, roi_prefix, explained_series, outpath):
    subj_traj = (
        df_in.groupby(["Subject", "condition_label", "reveal_idx"])[
            [f"{roi_prefix}_PC1", f"{roi_prefix}_PC2"]
        ]
        .mean()
        .reset_index()
    )

    grand = (
        subj_traj.groupby(["condition_label", "reveal_idx"])[
            [f"{roi_prefix}_PC1", f"{roi_prefix}_PC2"]
        ]
        .mean()
        .reset_index()
    )

    plt.figure(figsize=(8, 6))
    for cond in ["equal", "unequal"]:
        sub = grand[grand["condition_label"] == cond].sort_values("reveal_idx")
        if len(sub) == 0:
            continue

        plt.plot(
            sub[f"{roi_prefix}_PC1"],
            sub[f"{roi_prefix}_PC2"],
            marker="o",
            label=cond
        )
        for _, row in sub.iterrows():
            plt.text(
                row[f"{roi_prefix}_PC1"],
                row[f"{roi_prefix}_PC2"],
                str(int(row["reveal_idx"])),
                fontsize=9
            )

    plt.axhline(0, linewidth=0.8)
    plt.axvline(0, linewidth=0.8)
    plt.xlabel(f"{roi_prefix.upper()} PC1 ({explained_series.iloc[0]*100:.1f}%)")
    plt.ylabel(f"{roi_prefix.upper()} PC2 ({explained_series.iloc[1]*100:.1f}%)")
    plt.title(f"{roi_prefix.upper()} subject-centered trajectory")
    plt.legend()
    plt.tight_layout()
    plt.savefig(outpath, dpi=200)
    plt.close()

plot_roi_trajectory(
    df_state, "acc", acc_explained,
    os.path.join(OUTPUT_DIR, "acc_subject_centered_trajectory.png")
)
plot_roi_trajectory(
    df_state, "vmpfc", vmpfc_explained,
    os.path.join(OUTPUT_DIR, "vmpfc_subject_centered_trajectory.png")
)

# =========================================================
# trajectory geometry
# 每个 Subject x Game x condition 计算一条 trajectory 的几何量
# ROI 单独算
# =========================================================
def compute_geometry_metrics_for_group(g, pc_cols, prefix):
    g = g.sort_values("reveal_idx").copy()
    X = g[pc_cols].to_numpy(dtype=float)

    # step vectors
    if X.shape[0] < 2:
        return None

    step_vecs = np.diff(X, axis=0)
    step_sizes = np.sqrt(np.sum(step_vecs ** 2, axis=1))

    # total length
    traj_length = np.sum(step_sizes)

    # mean / max step
    mean_step = np.mean(step_sizes)
    max_step = np.max(step_sizes)

    # start-to-end distance
    displacement = np.sqrt(np.sum((X[-1] - X[0]) ** 2))

    # efficiency = displacement / length
    efficiency = displacement / traj_length if traj_length > 1e-8 else np.nan

    # curvature / turning angle
    turning_angles = []
    for i in range(len(step_vecs) - 1):
        a = step_vecs[i]
        b = step_vecs[i + 1]
        na = np.linalg.norm(a)
        nb = np.linalg.norm(b)
        if na < 1e-8 or nb < 1e-8:
            turning_angles.append(np.nan)
            continue
        cosang = np.dot(a, b) / (na * nb)
        cosang = np.clip(cosang, -1, 1)
        ang = np.arccos(cosang)
        turning_angles.append(ang)

    mean_turn_angle = np.nanmean(turning_angles) if len(turning_angles) > 0 else np.nan

    # terminal stability: distance to terminal state at each reveal
    terminal = X[-1]
    dist_to_terminal = np.sqrt(np.sum((X - terminal) ** 2, axis=1))
    mean_dist_to_terminal = np.mean(dist_to_terminal[:-1]) if len(dist_to_terminal) > 1 else np.nan

    out = {
        f"{prefix}_traj_length": traj_length,
        f"{prefix}_mean_step": mean_step,
        f"{prefix}_max_step": max_step,
        f"{prefix}_displacement": displacement,
        f"{prefix}_efficiency": efficiency,
        f"{prefix}_mean_turn_angle": mean_turn_angle,
        f"{prefix}_mean_dist_to_terminal": mean_dist_to_terminal,
    }
    return out

geometry_rows = []
for (subj, game, cond), g in df_state.groupby(["Subject", "Game", "condition_label"]):
    row = {
        "Subject": subj,
        "Game": game,
        "condition_label": cond,
    }

    acc_metrics = compute_geometry_metrics_for_group(
        g, ["acc_PC1", "acc_PC2", "acc_PC3"], "acc"
    )
    vmpfc_metrics = compute_geometry_metrics_for_group(
        g, ["vmpfc_PC1", "vmpfc_PC2", "vmpfc_PC3"], "vmpfc"
    )

    if acc_metrics is not None:
        row.update(acc_metrics)
    if vmpfc_metrics is not None:
        row.update(vmpfc_metrics)

    geometry_rows.append(row)

geometry_df = pd.DataFrame(geometry_rows)
geometry_df.to_csv(os.path.join(OUTPUT_DIR, "trajectory_geometry_metrics.csv"), index=False)

# =========================================================
# geometry group summary
# =========================================================
geom_summary = geometry_df.groupby("condition_label").mean(numeric_only=True)
geom_summary.to_csv(os.path.join(OUTPUT_DIR, "trajectory_geometry_condition_means.csv"))

# subject-level paired stats via clustered OLS / simple mixed
geom_stats_txt = []
geom_metric_cols = [
    c for c in geometry_df.columns
    if c not in ["Subject", "Game", "condition_label"]
]

for metric in geom_metric_cols:
    subdf = geometry_df[["Subject", "condition_label", metric]].dropna().copy()
    if subdf[metric].nunique() < 3:
        continue

    try:
        model = smf.ols(f"{metric} ~ C(condition_label)", data=subdf).fit(
            cov_type="cluster",
            cov_kwds={"groups": subdf["Subject"]}
        )
        geom_stats_txt.append(f"\n=== {metric} ===\n")
        geom_stats_txt.append(str(model.summary()))
    except Exception as e:
        geom_stats_txt.append(f"\n=== {metric} ===\n")
        geom_stats_txt.append(f"FAILED: {e}")

with open(os.path.join(OUTPUT_DIR, "trajectory_geometry_stats.txt"), "w") as f:
    f.write("\n".join(geom_stats_txt))

# =========================================================
# representation emergence
# 思路：
# reveal-by-reveal, ROI-state -> task variable
# 分类变量: logistic AUC
# 连续变量: CV R2
# ROI 分开跑
# =========================================================
def is_binary_series(s):
    vals = pd.Series(s).dropna().unique()
    if len(vals) <= 2:
        return True
    if set(vals).issubset({0, 1, False, True}):
        return True
    return False

def cross_validated_decode(X, y, binary=True, n_splits=5):
    if len(y) < 20:
        return np.nan

    if binary:
        y2 = pd.Series(y).astype(int).to_numpy()
        if len(np.unique(y2)) < 2:
            return np.nan
        clf = LogisticRegression(max_iter=2000)
        cv = StratifiedKFold(n_splits=min(n_splits, np.min(np.bincount(y2))), shuffle=True, random_state=42)
        scores = cross_val_score(clf, X, y2, cv=cv, scoring="roc_auc")
        return np.mean(scores)
    else:
        y2 = pd.Series(y).astype(float).to_numpy()
        if np.nanstd(y2) < 1e-8:
            return np.nan
        reg = LinearRegression()
        cv = KFold(n_splits=n_splits, shuffle=True, random_state=42)
        scores = cross_val_score(reg, X, y2, cv=cv, scoring="r2")
        return np.mean(scores)

repr_rows = []

roi_map = {
    "acc": ["acc_PC1", "acc_PC2", "acc_PC3"],
    "vmpfc": ["vmpfc_PC1", "vmpfc_PC2", "vmpfc_PC3"],
    "joint": ["acc_PC1", "acc_PC2", "acc_PC3", "vmpfc_PC1", "vmpfc_PC2", "vmpfc_PC3"],
}

for reveal in sorted(df_state["reveal_idx"].unique()):
    sub_r = df_state[df_state["reveal_idx"] == reveal].copy()

    for target in available_targets:
        yt = sub_r[target]
        binary = is_binary_series(yt)

        valid_mask = yt.notna().copy()
        for roi_name, roi_cols in roi_map.items():
            Xm = sub_r.loc[valid_mask, roi_cols].to_numpy(dtype=float)
            ym = yt.loc[valid_mask].to_numpy()

            if len(ym) < 20:
                score = np.nan
            else:
                score = cross_validated_decode(Xm, ym, binary=binary, n_splits=5)

            repr_rows.append({
                "reveal_idx": reveal,
                "target": target,
                "roi": roi_name,
                "target_type": "binary" if binary else "continuous",
                "cv_score": score
            })

repr_df = pd.DataFrame(repr_rows)
repr_df.to_csv(os.path.join(OUTPUT_DIR, "representation_emergence_scores.csv"), index=False)

# 画几张重点图
def plot_repr_target(repr_df, target_name, outpath):
    sub = repr_df[repr_df["target"] == target_name].copy()
    if len(sub) == 0:
        return

    plt.figure(figsize=(8, 5))
    for roi_name in ["acc", "vmpfc", "joint"]:
        ss = sub[sub["roi"] == roi_name].sort_values("reveal_idx")
        if len(ss) == 0:
            continue
        plt.plot(ss["reveal_idx"], ss["cv_score"], marker="o", label=roi_name)

    ylabel = "CV ROC-AUC / CV R2"
    plt.xlabel("Reveal index")
    plt.ylabel(ylabel)
    plt.title(f"Representation emergence: {target_name}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(outpath, dpi=200)
    plt.close()

priority_targets = [
    "best_deck_snapshot",
    "value_gap_snapshot",
    "sampling_imbalance_after",
    "current_deck_was_least_sampled_before",
    "current_reward",
]

for tgt in priority_targets:
    if tgt in available_targets:
        plot_repr_target(
            repr_df, tgt,
            os.path.join(OUTPUT_DIR, f"repr_emergence_{tgt}.png")
        )

# =========================================================
# emergence slope stats
# 每个 target x roi 看 cv_score 是否随 reveal 增大
# 这里是描述性：对 reveal_idx -> cv_score 做简单线性拟合
# =========================================================
emergence_stats = []
for (target, roi), g in repr_df.groupby(["target", "roi"]):
    g = g.dropna(subset=["cv_score"]).sort_values("reveal_idx")
    if len(g) < 3:
        continue
    x = g["reveal_idx"].to_numpy(dtype=float)
    y = g["cv_score"].to_numpy(dtype=float)
    coef = np.polyfit(x, y, deg=1)[0]
    emergence_stats.append({
        "target": target,
        "roi": roi,
        "emergence_slope": coef,
        "mean_score": np.mean(y),
        "max_score": np.max(y),
    })

emergence_stats_df = pd.DataFrame(emergence_stats)
emergence_stats_df.to_csv(os.path.join(OUTPUT_DIR, "representation_emergence_slopes.csv"), index=False)

# =========================================================
# terminal state analysis
# 终态 = reveal 6
# 1) decode condition
# 2) decode best_deck_snapshot
# 3) predict value_gap_snapshot
# 4) terminal dispersion by condition
# =========================================================
terminal = df_state[df_state["reveal_idx"] == 6].copy()

terminal_results = []

def run_terminal_decode(df_term, roi_name, roi_cols, y, y_name, binary=True):
    valid = pd.Series(y).notna().values
    X = df_term.loc[valid, roi_cols].to_numpy(dtype=float)
    yy = pd.Series(y).loc[valid].to_numpy()

    score = cross_validated_decode(X, yy, binary=binary, n_splits=5)
    return {
        "analysis": y_name,
        "roi": roi_name,
        "score": score,
        "score_type": "ROC-AUC" if binary else "CV_R2"
    }

# 1) decode condition
y_cond = (terminal["condition_label"] == "unequal").astype(int)
for roi_name, roi_cols in roi_map.items():
    terminal_results.append(
        run_terminal_decode(terminal, roi_name, roi_cols, y_cond, "decode_condition", binary=True)
    )

# 2) decode best deck
if "best_deck_snapshot" in terminal.columns:
    y_best = terminal["best_deck_snapshot"]
    if is_binary_series(y_best):
        for roi_name, roi_cols in roi_map.items():
            terminal_results.append(
                run_terminal_decode(terminal, roi_name, roi_cols, y_best, "decode_best_deck_snapshot", binary=True)
            )

# 3) predict value gap
if "value_gap_snapshot" in terminal.columns:
    y_gap = terminal["value_gap_snapshot"]
    for roi_name, roi_cols in roi_map.items():
        terminal_results.append(
            run_terminal_decode(terminal, roi_name, roi_cols, y_gap, "predict_value_gap_snapshot", binary=False)
        )

terminal_results_df = pd.DataFrame(terminal_results)
terminal_results_df.to_csv(os.path.join(OUTPUT_DIR, "terminal_state_decode_results.csv"), index=False)

# 4) terminal dispersion by condition
# 每个 condition 内，终态围绕 condition centroid 的距离
dispersion_rows = []
for roi_name, roi_cols in roi_map.items():
    for cond, g in terminal.groupby("condition_label"):
        X = g[roi_cols].to_numpy(dtype=float)
        centroid = X.mean(axis=0, keepdims=True)
        d = np.sqrt(np.sum((X - centroid) ** 2, axis=1))
        for i, val in enumerate(d):
            dispersion_rows.append({
                "roi": roi_name,
                "condition_label": cond,
                "dispersion": val,
                "Subject": g.iloc[i]["Subject"],
                "Game": g.iloc[i]["Game"],
            })

dispersion_df = pd.DataFrame(dispersion_rows)
dispersion_df.to_csv(os.path.join(OUTPUT_DIR, "terminal_state_dispersion.csv"), index=False)

dispersion_stats_txt = []
for roi_name in dispersion_df["roi"].unique():
    sub = dispersion_df[dispersion_df["roi"] == roi_name].copy()
    try:
        model = smf.ols("dispersion ~ C(condition_label)", data=sub).fit(
            cov_type="cluster",
            cov_kwds={"groups": sub["Subject"]}
        )
        dispersion_stats_txt.append(f"\n=== terminal dispersion: {roi_name} ===\n")
        dispersion_stats_txt.append(str(model.summary()))
    except Exception as e:
        dispersion_stats_txt.append(f"\n=== terminal dispersion: {roi_name} ===\nFAILED: {e}\n")

with open(os.path.join(OUTPUT_DIR, "terminal_dispersion_stats.txt"), "w") as f:
    f.write("\n".join(dispersion_stats_txt))

# =========================================================
# reveal-to-terminal distance
# 看每个 reveal 离本 game 终态有多远，是否更快收敛
# =========================================================
dist_rows = []
for (subj, game, cond), g in df_state.groupby(["Subject", "Game", "condition_label"]):
    g = g.sort_values("reveal_idx").copy()

    for roi_name, roi_cols in roi_map.items():
        X = g[roi_cols].to_numpy(dtype=float)
        term = X[-1]
        d = np.sqrt(np.sum((X - term) ** 2, axis=1))

        for rr, dd in zip(g["reveal_idx"].values, d):
            dist_rows.append({
                "Subject": subj,
                "Game": game,
                "condition_label": cond,
                "reveal_idx": rr,
                "roi": roi_name,
                "distance_to_terminal": dd
            })

dist_to_terminal_df = pd.DataFrame(dist_rows)
dist_to_terminal_df.to_csv(os.path.join(OUTPUT_DIR, "distance_to_terminal_by_reveal.csv"), index=False)

# plot distance-to-terminal
for roi_name in ["acc", "vmpfc", "joint"]:
    sub = dist_to_terminal_df[dist_to_terminal_df["roi"] == roi_name].copy()
    mean_df = (
        sub.groupby(["condition_label", "reveal_idx"])["distance_to_terminal"]
        .mean()
        .reset_index()
    )

    plt.figure(figsize=(7, 5))
    for cond in ["equal", "unequal"]:
        ss = mean_df[mean_df["condition_label"] == cond].sort_values("reveal_idx")
        if len(ss) == 0:
            continue
        plt.plot(ss["reveal_idx"], ss["distance_to_terminal"], marker="o", label=cond)

    plt.xlabel("Reveal index")
    plt.ylabel("Distance to terminal state")
    plt.title(f"{roi_name.upper()} convergence to terminal state")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"{roi_name}_distance_to_terminal.png"), dpi=200)
    plt.close()

# =========================================================
# ROI coupling / divergence over reveal
# 用 acc-vmpfc PC 差值看是否后期分离或收敛
# =========================================================
roi_div_summary = (
    df_state.groupby(["condition_label", "reveal_idx"])[
        ["roi_pc1_diff", "roi_pc2_diff", "roi_pc3_diff"]
    ]
    .mean()
    .reset_index()
)
roi_div_summary.to_csv(os.path.join(OUTPUT_DIR, "roi_divergence_summary.csv"), index=False)

for diff_col in ["roi_pc1_diff", "roi_pc2_diff", "roi_pc3_diff"]:
    plt.figure(figsize=(7, 5))
    for cond in ["equal", "unequal"]:
        sub = roi_div_summary[roi_div_summary["condition_label"] == cond].sort_values("reveal_idx")
        if len(sub) == 0:
            continue
        plt.plot(sub["reveal_idx"], sub[diff_col], marker="o", label=cond)

    plt.xlabel("Reveal index")
    plt.ylabel(diff_col)
    plt.title(f"ACC-vmPFC divergence: {diff_col}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"{diff_col}_trajectory.png"), dpi=200)
    plt.close()

# =========================================================
# save full state table
# =========================================================
df_state.to_csv(os.path.join(OUTPUT_DIR, "reveal_level_with_roi_state_scores.csv"), index=False)

# =========================================================
# concise text summary scaffold
# =========================================================
summary_txt = []
summary_txt.append("ACC explained variance:")
summary_txt.append(str(acc_explained))
summary_txt.append("\nvmPFC explained variance:")
summary_txt.append(str(vmpfc_explained))

summary_txt.append("\n\nGeometry condition means:")
summary_txt.append(str(geom_summary))

summary_txt.append("\n\nTerminal decode results:")
summary_txt.append(str(terminal_results_df))

summary_txt.append("\n\nRepresentation emergence slopes:")
summary_txt.append(str(emergence_stats_df.sort_values(['target', 'roi']).head(50)))

with open(os.path.join(OUTPUT_DIR, "quick_summary.txt"), "w") as f:
    f.write("\n".join(summary_txt))

print("\nDone.")
print("Outputs saved to:", OUTPUT_DIR)

Loaded: (10362, 195)
After complete-game filter: (10362, 195)
ACC feature n = 40
vmPFC feature n = 40
Available targets: ['current_reward', 'best_deck_snapshot', 'value_gap_snapshot', 'imbalance_change', 'current_deck_was_least_sampled_before', 'current_deck_is_best_snapshot']

Done.
Outputs saved to: /root/autodl-tmp/reveal_roi_state_space_results/


In [ ]:
import os
import shutil
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.model_selection import StratifiedKFold, KFold, cross_val_score
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore")

# =========================================================
# paths
# =========================================================
INPUT_CSV = "/root/autodl-tmp/reveal_neural_feature_tables/reveal_level_neural_feature_table.csv"
OUTPUT_DIR = "/root/autodl-tmp/reveal_roi_state_space_results/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 备份旧结果（可选，但建议保留）
BACKUP_DIR = os.path.join(OUTPUT_DIR, "_backup_old_wrong_outputs")
if not os.path.exists(BACKUP_DIR):
    os.makedirs(BACKUP_DIR, exist_ok=True)
    for fn in os.listdir(OUTPUT_DIR):
        src = os.path.join(OUTPUT_DIR, fn)
        dst = os.path.join(BACKUP_DIR, fn)
        if os.path.isfile(src):
            try:
                shutil.copy2(src, dst)
            except Exception:
                pass

# =========================================================
# load
# =========================================================
df = pd.read_csv(INPUT_CSV)
print("Loaded:", df.shape)

# 只保留完整 6-reveal game
df = df[df["game_complete_6"] == True].copy()
df = df.sort_values(["Subject", "Game", "reveal_idx"]).reset_index(drop=True)
print("After complete-game filter:", df.shape)

# =========================================================
# helper: pick feature columns by ROI
# =========================================================
def pick_roi_features(columns, roi_name):
    feats = []
    for c in columns:
        if not c.startswith(f"{roi_name}_"):
            continue
        if (
            "_bin" in c
            or c.endswith("_full_mean")
            or c.endswith("_full_auc")
            or c.endswith("_channel_sd_full")
        ):
            feats.append(c)
    return feats

acc_cols = pick_roi_features(df.columns, "acc")
vmpfc_cols = pick_roi_features(df.columns, "vmpfc")

# 去掉全空列
acc_cols = [c for c in acc_cols if df[c].notna().sum() > 0]
vmpfc_cols = [c for c in vmpfc_cols if df[c].notna().sum() > 0]

print("ACC feature n =", len(acc_cols))
print("vmPFC feature n =", len(vmpfc_cols))

# =========================================================
# sanity check
# =========================================================
required_cols = ["Subject", "Game", "condition_label", "reveal_idx"]
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

candidate_targets = [
    "current_reward",
    "best_deck_snapshot",
    "value_gap_snapshot",
    "sampling_imbalance_after",
    "sampling_imbalance_before",
    "imbalance_change",
    "current_deck_was_least_sampled_before",
    "current_deck_is_best_snapshot",
]

available_targets = [c for c in candidate_targets if c in df.columns]
print("Available targets:", available_targets)

# =========================================================
# helper: identify ROI-valid subjects / rows
# 关键修正：一个 ROI 的 PCA 只能用真正有该 ROI 数据的被试
# =========================================================
def get_valid_subjects_for_roi(df_in, feature_cols):
    if len(feature_cols) == 0:
        return []

    subj_valid = df_in.groupby("Subject")[feature_cols].apply(
        lambda x: x.notna().any().any()
    )
    valid_subjects = subj_valid[subj_valid].index.tolist()
    return valid_subjects

def get_valid_row_mask(df_in, feature_cols):
    if len(feature_cols) == 0:
        return pd.Series(False, index=df_in.index)

    return df_in[feature_cols].notna().any(axis=1)

acc_valid_subjects = get_valid_subjects_for_roi(df, acc_cols)
vmpfc_valid_subjects = get_valid_subjects_for_roi(df, vmpfc_cols)

print("ACC valid subjects =", len(acc_valid_subjects))
print("vmPFC valid subjects =", len(vmpfc_valid_subjects))

df["has_acc_data"] = df["Subject"].isin(acc_valid_subjects).astype(int)
df["has_vmpfc_data"] = df["Subject"].isin(vmpfc_valid_subjects).astype(int)

# =========================================================
# helper: preprocessing
# 注意：这里只对 ROI-valid 子集做 preprocessing
# =========================================================
def preprocess_matrix(X):
    imputer = SimpleImputer(strategy="median")
    X_imp = imputer.fit_transform(X)

    scaler = StandardScaler()
    X_std = scaler.fit_transform(X_imp)
    return X_std, imputer, scaler

# =========================================================
# helper: centering
# =========================================================
def center_features(df_in, feature_cols, mode="subject_centered"):
    X = df_in[feature_cols].copy()

    if mode == "raw":
        return X

    out = X.copy()

    if mode == "subject_centered":
        subj_means = df_in.groupby("Subject")[feature_cols].transform("mean")
        out = X - subj_means

    elif mode == "subject_condition_centered":
        sc_means = df_in.groupby(["Subject", "condition_label"])[feature_cols].transform("mean")
        out = X - sc_means

    else:
        raise ValueError(f"Unknown center mode: {mode}")

    return out

# =========================================================
# helper: fit PCA for one ROI
# 关键修正：
#   1) 只在该 ROI-valid 子集上拟合
#   2) 返回的 score 只包含该子集行
# =========================================================
def run_roi_pca(df_in, roi_name, feature_cols, center_mode="subject_centered", n_components=3):
    if len(feature_cols) == 0:
        raise ValueError(f"No features for ROI={roi_name}")

    # 先筛行：只保留该 ROI 有任意非空特征的行
    valid_row_mask = get_valid_row_mask(df_in, feature_cols)
    df_roi = df_in.loc[valid_row_mask].copy()

    if len(df_roi) == 0:
        raise ValueError(f"No valid rows for ROI={roi_name}")

    X_centered = center_features(df_roi, feature_cols, mode=center_mode)
    X_std, imputer, scaler = preprocess_matrix(X_centered.to_numpy(dtype=float))

    pca = PCA(n_components=n_components)
    X_pca = pca.fit_transform(X_std)

    key_cols = ["Subject", "Game", "condition_label", "reveal_idx"]
    if "Condition" in df_roi.columns:
        key_cols.append("Condition")

    score_df = df_roi[key_cols].copy().reset_index(drop=True)
    for i in range(n_components):
        score_df[f"{roi_name}_PC{i+1}"] = X_pca[:, i]

    loadings = pd.DataFrame(
        pca.components_.T,
        index=feature_cols,
        columns=[f"{roi_name}_PC{i+1}" for i in range(n_components)]
    )

    explained = pd.Series(
        pca.explained_variance_ratio_,
        index=[f"{roi_name}_PC{i+1}" for i in range(n_components)],
        name="explained_variance_ratio"
    )

    return score_df, loadings, explained, pca, imputer, scaler, df_roi

# =========================================================
# run ROI-specific PCA
# =========================================================
acc_scores, acc_loadings, acc_explained, acc_pca, _, _, df_acc = run_roi_pca(
    df[df["has_acc_data"] == 1].copy(),
    "acc",
    acc_cols,
    center_mode="subject_centered",
    n_components=3
)

vmpfc_scores, vmpfc_loadings, vmpfc_explained, vmpfc_pca, _, _, df_vmpfc = run_roi_pca(
    df[df["has_vmpfc_data"] == 1].copy(),
    "vmpfc",
    vmpfc_cols,
    center_mode="subject_centered",
    n_components=3
)

# 保存 PCA 输出（覆盖旧文件）
acc_loadings.to_csv(os.path.join(OUTPUT_DIR, "acc_pca_loadings.csv"))
vmpfc_loadings.to_csv(os.path.join(OUTPUT_DIR, "vmpfc_pca_loadings.csv"))
acc_explained.to_csv(os.path.join(OUTPUT_DIR, "acc_explained_variance.csv"), header=True)
vmpfc_explained.to_csv(os.path.join(OUTPUT_DIR, "vmpfc_explained_variance.csv"), header=True)

# =========================================================
# merge back safely
# 关键修正：left merge，不再按 .values 硬贴
# =========================================================
df_state = df.copy()

merge_keys = ["Subject", "Game", "condition_label", "reveal_idx"]
if "Condition" in df_state.columns and "Condition" in acc_scores.columns and "Condition" in vmpfc_scores.columns:
    merge_keys = ["Subject", "Game", "Condition", "condition_label", "reveal_idx"]

df_state = df_state.merge(
    acc_scores[merge_keys + ["acc_PC1", "acc_PC2", "acc_PC3"]],
    on=merge_keys,
    how="left"
)

df_state = df_state.merge(
    vmpfc_scores[merge_keys + ["vmpfc_PC1", "vmpfc_PC2", "vmpfc_PC3"]],
    on=merge_keys,
    how="left"
)

# ROI divergence / coupling proxy
for i in [1, 2, 3]:
    df_state[f"roi_pc{i}_diff"] = df_state[f"acc_PC{i}"] - df_state[f"vmpfc_PC{i}"]

# =========================================================
# plot subject-averaged ROI trajectories
# 只用该 ROI 有效数据
# =========================================================
def plot_roi_trajectory(df_in, roi_prefix, explained_series, outpath, valid_flag_col):
    sub0 = df_in[df_in[valid_flag_col] == 1].copy()
    sub0 = sub0.dropna(subset=[f"{roi_prefix}_PC1", f"{roi_prefix}_PC2"])

    if len(sub0) == 0:
        return

    subj_traj = (
        sub0.groupby(["Subject", "condition_label", "reveal_idx"])[
            [f"{roi_prefix}_PC1", f"{roi_prefix}_PC2"]
        ]
        .mean()
        .reset_index()
    )

    grand = (
        subj_traj.groupby(["condition_label", "reveal_idx"])[
            [f"{roi_prefix}_PC1", f"{roi_prefix}_PC2"]
        ]
        .mean()
        .reset_index()
    )

    plt.figure(figsize=(8, 6))
    for cond in ["equal", "unequal"]:
        sub = grand[grand["condition_label"] == cond].sort_values("reveal_idx")
        if len(sub) == 0:
            continue

        plt.plot(
            sub[f"{roi_prefix}_PC1"],
            sub[f"{roi_prefix}_PC2"],
            marker="o",
            label=cond
        )
        for _, row in sub.iterrows():
            plt.text(
                row[f"{roi_prefix}_PC1"],
                row[f"{roi_prefix}_PC2"],
                str(int(row["reveal_idx"])),
                fontsize=9
            )

    plt.axhline(0, linewidth=0.8)
    plt.axvline(0, linewidth=0.8)
    plt.xlabel(f"{roi_prefix.upper()} PC1 ({explained_series.iloc[0]*100:.1f}%)")
    plt.ylabel(f"{roi_prefix.upper()} PC2 ({explained_series.iloc[1]*100:.1f}%)")
    plt.title(f"{roi_prefix.upper()} subject-centered trajectory")
    plt.legend()
    plt.tight_layout()
    plt.savefig(outpath, dpi=200)
    plt.close()

plot_roi_trajectory(
    df_state, "acc", acc_explained,
    os.path.join(OUTPUT_DIR, "acc_subject_centered_trajectory.png"),
    valid_flag_col="has_acc_data"
)

plot_roi_trajectory(
    df_state, "vmpfc", vmpfc_explained,
    os.path.join(OUTPUT_DIR, "vmpfc_subject_centered_trajectory.png"),
    valid_flag_col="has_vmpfc_data"
)

# =========================================================
# trajectory geometry
# 关键修正：每个 ROI 只在该 ROI-valid 数据上计算
# joint 只在 ACC+vmPFC 都存在时计算
# =========================================================
def compute_geometry_metrics_for_group(g, pc_cols, prefix):
    g = g.sort_values("reveal_idx").copy()
    g = g.dropna(subset=pc_cols)

    X = g[pc_cols].to_numpy(dtype=float)
    if X.shape[0] < 2:
        return None

    step_vecs = np.diff(X, axis=0)
    step_sizes = np.sqrt(np.sum(step_vecs ** 2, axis=1))

    traj_length = np.sum(step_sizes)
    mean_step = np.mean(step_sizes)
    max_step = np.max(step_sizes)

    displacement = np.sqrt(np.sum((X[-1] - X[0]) ** 2))
    efficiency = displacement / traj_length if traj_length > 1e-8 else np.nan

    turning_angles = []
    for i in range(len(step_vecs) - 1):
        a = step_vecs[i]
        b = step_vecs[i + 1]
        na = np.linalg.norm(a)
        nb = np.linalg.norm(b)
        if na < 1e-8 or nb < 1e-8:
            turning_angles.append(np.nan)
            continue
        cosang = np.dot(a, b) / (na * nb)
        cosang = np.clip(cosang, -1, 1)
        ang = np.arccos(cosang)
        turning_angles.append(ang)

    mean_turn_angle = np.nanmean(turning_angles) if len(turning_angles) > 0 else np.nan

    terminal = X[-1]
    dist_to_terminal = np.sqrt(np.sum((X - terminal) ** 2, axis=1))
    mean_dist_to_terminal = np.mean(dist_to_terminal[:-1]) if len(dist_to_terminal) > 1 else np.nan

    out = {
        f"{prefix}_traj_length": traj_length,
        f"{prefix}_mean_step": mean_step,
        f"{prefix}_max_step": max_step,
        f"{prefix}_displacement": displacement,
        f"{prefix}_efficiency": efficiency,
        f"{prefix}_mean_turn_angle": mean_turn_angle,
        f"{prefix}_mean_dist_to_terminal": mean_dist_to_terminal,
    }
    return out

geometry_rows = []
for (subj, game, cond), g in df_state.groupby(["Subject", "Game", "condition_label"]):
    row = {
        "Subject": subj,
        "Game": game,
        "condition_label": cond,
        "has_acc_data": int(g["has_acc_data"].iloc[0]),
        "has_vmpfc_data": int(g["has_vmpfc_data"].iloc[0]),
    }

    if row["has_acc_data"] == 1:
        acc_metrics = compute_geometry_metrics_for_group(
            g, ["acc_PC1", "acc_PC2", "acc_PC3"], "acc"
        )
        if acc_metrics is not None:
            row.update(acc_metrics)

    if row["has_vmpfc_data"] == 1:
        vmpfc_metrics = compute_geometry_metrics_for_group(
            g, ["vmpfc_PC1", "vmpfc_PC2", "vmpfc_PC3"], "vmpfc"
        )
        if vmpfc_metrics is not None:
            row.update(vmpfc_metrics)

    if row["has_acc_data"] == 1 and row["has_vmpfc_data"] == 1:
        joint_metrics = compute_geometry_metrics_for_group(
            g,
            ["acc_PC1", "acc_PC2", "acc_PC3", "vmpfc_PC1", "vmpfc_PC2", "vmpfc_PC3"],
            "joint"
        )
        if joint_metrics is not None:
            row.update(joint_metrics)

    geometry_rows.append(row)

geometry_df = pd.DataFrame(geometry_rows)
geometry_df.to_csv(os.path.join(OUTPUT_DIR, "trajectory_geometry_metrics.csv"), index=False)

geom_summary = geometry_df.groupby("condition_label").mean(numeric_only=True)
geom_summary.to_csv(os.path.join(OUTPUT_DIR, "trajectory_geometry_condition_means.csv"))

geom_stats_txt = []
geom_metric_cols = [
    c for c in geometry_df.columns
    if c not in ["Subject", "Game", "condition_label", "has_acc_data", "has_vmpfc_data"]
]

for metric in geom_metric_cols:
    subdf = geometry_df[["Subject", "condition_label", metric]].dropna().copy()
    if len(subdf) < 10 or subdf[metric].nunique() < 3:
        continue

    try:
        model = smf.ols(f"{metric} ~ C(condition_label)", data=subdf).fit(
            cov_type="cluster",
            cov_kwds={"groups": subdf["Subject"]}
        )
        geom_stats_txt.append(f"\n=== {metric} ===\n")
        geom_stats_txt.append(str(model.summary()))
    except Exception as e:
        geom_stats_txt.append(f"\n=== {metric} ===\nFAILED: {e}\n")

with open(os.path.join(OUTPUT_DIR, "trajectory_geometry_stats.txt"), "w") as f:
    f.write("\n".join(geom_stats_txt))

# =========================================================
# representation emergence
# 关键修正：每个 ROI 都只在自己有效的行上跑
# joint 只在两者都有效的行上跑
# =========================================================
def is_binary_series(s):
    vals = pd.Series(s).dropna().unique()
    if len(vals) <= 2:
        return True
    if set(vals).issubset({0, 1, False, True}):
        return True
    return False

def cross_validated_decode(X, y, binary=True, n_splits=5):
    if len(y) < 20:
        return np.nan

    if binary:
        y2 = pd.Series(y).astype(int).to_numpy()
        if len(np.unique(y2)) < 2:
            return np.nan

        class_counts = np.bincount(y2)
        if len(class_counts) < 2 or np.min(class_counts) < 2:
            return np.nan

        nfold = min(n_splits, int(np.min(class_counts)))
        if nfold < 2:
            return np.nan

        clf = LogisticRegression(max_iter=2000)
        cv = StratifiedKFold(n_splits=nfold, shuffle=True, random_state=42)
        scores = cross_val_score(clf, X, y2, cv=cv, scoring="roc_auc")
        return np.mean(scores)

    else:
        y2 = pd.Series(y).astype(float).to_numpy()
        if np.nanstd(y2) < 1e-8:
            return np.nan

        nfold = min(n_splits, len(y2))
        if nfold < 2:
            return np.nan

        reg = LinearRegression()
        cv = KFold(n_splits=nfold, shuffle=True, random_state=42)
        scores = cross_val_score(reg, X, y2, cv=cv, scoring="r2")
        return np.mean(scores)

repr_rows = []

for reveal in sorted(df_state["reveal_idx"].unique()):
    sub_r = df_state[df_state["reveal_idx"] == reveal].copy()

    for target in available_targets:
        yt = sub_r[target]
        binary = is_binary_series(yt)

        # ACC
        valid_acc = (
            yt.notna()
            & sub_r["has_acc_data"].eq(1)
            & sub_r[["acc_PC1", "acc_PC2", "acc_PC3"]].notna().all(axis=1)
        )
        Xa = sub_r.loc[valid_acc, ["acc_PC1", "acc_PC2", "acc_PC3"]].to_numpy(dtype=float)
        ya = yt.loc[valid_acc].to_numpy()
        score_acc = cross_validated_decode(Xa, ya, binary=binary, n_splits=5) if len(ya) >= 20 else np.nan

        repr_rows.append({
            "reveal_idx": reveal,
            "target": target,
            "roi": "acc",
            "target_type": "binary" if binary else "continuous",
            "cv_score": score_acc,
            "n_rows": len(ya),
        })

        # vmPFC
        valid_v = (
            yt.notna()
            & sub_r["has_vmpfc_data"].eq(1)
            & sub_r[["vmpfc_PC1", "vmpfc_PC2", "vmpfc_PC3"]].notna().all(axis=1)
        )
        Xv = sub_r.loc[valid_v, ["vmpfc_PC1", "vmpfc_PC2", "vmpfc_PC3"]].to_numpy(dtype=float)
        yv = yt.loc[valid_v].to_numpy()
        score_v = cross_validated_decode(Xv, yv, binary=binary, n_splits=5) if len(yv) >= 20 else np.nan

        repr_rows.append({
            "reveal_idx": reveal,
            "target": target,
            "roi": "vmpfc",
            "target_type": "binary" if binary else "continuous",
            "cv_score": score_v,
            "n_rows": len(yv),
        })

        # joint
        valid_j = (
            yt.notna()
            & sub_r["has_acc_data"].eq(1)
            & sub_r["has_vmpfc_data"].eq(1)
            & sub_r[["acc_PC1", "acc_PC2", "acc_PC3", "vmpfc_PC1", "vmpfc_PC2", "vmpfc_PC3"]].notna().all(axis=1)
        )
        Xj = sub_r.loc[valid_j, ["acc_PC1", "acc_PC2", "acc_PC3", "vmpfc_PC1", "vmpfc_PC2", "vmpfc_PC3"]].to_numpy(dtype=float)
        yj = yt.loc[valid_j].to_numpy()
        score_j = cross_validated_decode(Xj, yj, binary=binary, n_splits=5) if len(yj) >= 20 else np.nan

        repr_rows.append({
            "reveal_idx": reveal,
            "target": target,
            "roi": "joint",
            "target_type": "binary" if binary else "continuous",
            "cv_score": score_j,
            "n_rows": len(yj),
        })

repr_df = pd.DataFrame(repr_rows)
repr_df.to_csv(os.path.join(OUTPUT_DIR, "representation_emergence_scores.csv"), index=False)

def plot_repr_target(repr_df, target_name, outpath):
    sub = repr_df[repr_df["target"] == target_name].copy()
    if len(sub) == 0:
        return

    plt.figure(figsize=(8, 5))
    for roi_name in ["acc", "vmpfc", "joint"]:
        ss = sub[sub["roi"] == roi_name].sort_values("reveal_idx")
        if len(ss) == 0:
            continue
        plt.plot(ss["reveal_idx"], ss["cv_score"], marker="o", label=roi_name)

    plt.xlabel("Reveal index")
    plt.ylabel("CV ROC-AUC / CV R2")
    plt.title(f"Representation emergence: {target_name}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(outpath, dpi=200)
    plt.close()

priority_targets = [
    "best_deck_snapshot",
    "value_gap_snapshot",
    "sampling_imbalance_after",
    "current_deck_was_least_sampled_before",
    "current_reward",
]

for tgt in priority_targets:
    if tgt in available_targets:
        plot_repr_target(
            repr_df, tgt,
            os.path.join(OUTPUT_DIR, f"repr_emergence_{tgt}.png")
        )

emergence_stats = []
for (target, roi), g in repr_df.groupby(["target", "roi"]):
    g = g.dropna(subset=["cv_score"]).sort_values("reveal_idx")
    if len(g) < 3:
        continue
    x = g["reveal_idx"].to_numpy(dtype=float)
    y = g["cv_score"].to_numpy(dtype=float)
    coef = np.polyfit(x, y, deg=1)[0]
    emergence_stats.append({
        "target": target,
        "roi": roi,
        "emergence_slope": coef,
        "mean_score": np.mean(y),
        "max_score": np.max(y),
    })

emergence_stats_df = pd.DataFrame(emergence_stats)
emergence_stats_df.to_csv(os.path.join(OUTPUT_DIR, "representation_emergence_slopes.csv"), index=False)

# =========================================================
# terminal state analysis
# 关键修正：按 ROI-valid row 分开跑
# =========================================================
terminal = df_state[df_state["reveal_idx"] == 6].copy()
terminal_results = []

def run_terminal_decode(df_term, roi_name, roi_cols, y, y_name, binary=True):
    valid = pd.Series(y).notna().values
    X = df_term.loc[valid, roi_cols].to_numpy(dtype=float)
    yy = pd.Series(y).loc[valid].to_numpy()

    score = cross_validated_decode(X, yy, binary=binary, n_splits=5)
    return {
        "analysis": y_name,
        "roi": roi_name,
        "score": score,
        "score_type": "ROC-AUC" if binary else "CV_R2",
        "n_rows": len(yy),
    }

# 1) decode condition
y_cond_all = (terminal["condition_label"] == "unequal").astype(int)

term_acc = terminal[
    (terminal["has_acc_data"] == 1)
    & terminal[["acc_PC1", "acc_PC2", "acc_PC3"]].notna().all(axis=1)
].copy()

term_vmpfc = terminal[
    (terminal["has_vmpfc_data"] == 1)
    & terminal[["vmpfc_PC1", "vmpfc_PC2", "vmpfc_PC3"]].notna().all(axis=1)
].copy()

term_joint = terminal[
    (terminal["has_acc_data"] == 1)
    & (terminal["has_vmpfc_data"] == 1)
    & terminal[["acc_PC1", "acc_PC2", "acc_PC3", "vmpfc_PC1", "vmpfc_PC2", "vmpfc_PC3"]].notna().all(axis=1)
].copy()

terminal_results.append(
    run_terminal_decode(term_acc, "acc", ["acc_PC1", "acc_PC2", "acc_PC3"],
                        (term_acc["condition_label"] == "unequal").astype(int),
                        "decode_condition", binary=True)
)
terminal_results.append(
    run_terminal_decode(term_vmpfc, "vmpfc", ["vmpfc_PC1", "vmpfc_PC2", "vmpfc_PC3"],
                        (term_vmpfc["condition_label"] == "unequal").astype(int),
                        "decode_condition", binary=True)
)
terminal_results.append(
    run_terminal_decode(term_joint, "joint", ["acc_PC1", "acc_PC2", "acc_PC3", "vmpfc_PC1", "vmpfc_PC2", "vmpfc_PC3"],
                        (term_joint["condition_label"] == "unequal").astype(int),
                        "decode_condition", binary=True)
)

# 2) decode best deck
if "best_deck_snapshot" in terminal.columns:
    if is_binary_series(terminal["best_deck_snapshot"]):
        terminal_results.append(
            run_terminal_decode(term_acc, "acc", ["acc_PC1", "acc_PC2", "acc_PC3"],
                                term_acc["best_deck_snapshot"], "decode_best_deck_snapshot", binary=True)
        )
        terminal_results.append(
            run_terminal_decode(term_vmpfc, "vmpfc", ["vmpfc_PC1", "vmpfc_PC2", "vmpfc_PC3"],
                                term_vmpfc["best_deck_snapshot"], "decode_best_deck_snapshot", binary=True)
        )
        terminal_results.append(
            run_terminal_decode(term_joint, "joint", ["acc_PC1", "acc_PC2", "acc_PC3", "vmpfc_PC1", "vmpfc_PC2", "vmpfc_PC3"],
                                term_joint["best_deck_snapshot"], "decode_best_deck_snapshot", binary=True)
        )

# 3) predict value gap
if "value_gap_snapshot" in terminal.columns:
    terminal_results.append(
        run_terminal_decode(term_acc, "acc", ["acc_PC1", "acc_PC2", "acc_PC3"],
                            term_acc["value_gap_snapshot"], "predict_value_gap_snapshot", binary=False)
    )
    terminal_results.append(
        run_terminal_decode(term_vmpfc, "vmpfc", ["vmpfc_PC1", "vmpfc_PC2", "vmpfc_PC3"],
                            term_vmpfc["value_gap_snapshot"], "predict_value_gap_snapshot", binary=False)
    )
    terminal_results.append(
        run_terminal_decode(term_joint, "joint", ["acc_PC1", "acc_PC2", "acc_PC3", "vmpfc_PC1", "vmpfc_PC2", "vmpfc_PC3"],
                            term_joint["value_gap_snapshot"], "predict_value_gap_snapshot", binary=False)
    )

terminal_results_df = pd.DataFrame(terminal_results)
terminal_results_df.to_csv(os.path.join(OUTPUT_DIR, "terminal_state_decode_results.csv"), index=False)

# 4) terminal dispersion by condition
dispersion_rows = []

roi_term_map = {
    "acc": (term_acc, ["acc_PC1", "acc_PC2", "acc_PC3"]),
    "vmpfc": (term_vmpfc, ["vmpfc_PC1", "vmpfc_PC2", "vmpfc_PC3"]),
    "joint": (term_joint, ["acc_PC1", "acc_PC2", "acc_PC3", "vmpfc_PC1", "vmpfc_PC2", "vmpfc_PC3"]),
}

for roi_name, (df_term_roi, roi_cols) in roi_term_map.items():
    for cond, g in df_term_roi.groupby("condition_label"):
        g = g.dropna(subset=roi_cols)
        if len(g) == 0:
            continue
        X = g[roi_cols].to_numpy(dtype=float)
        centroid = X.mean(axis=0, keepdims=True)
        d = np.sqrt(np.sum((X - centroid) ** 2, axis=1))
        for ii, val in enumerate(d):
            dispersion_rows.append({
                "roi": roi_name,
                "condition_label": cond,
                "dispersion": val,
                "Subject": g.iloc[ii]["Subject"],
                "Game": g.iloc[ii]["Game"],
            })

dispersion_df = pd.DataFrame(dispersion_rows)
dispersion_df.to_csv(os.path.join(OUTPUT_DIR, "terminal_state_dispersion.csv"), index=False)

dispersion_stats_txt = []
for roi_name in dispersion_df["roi"].unique():
    sub = dispersion_df[dispersion_df["roi"] == roi_name].copy()
    try:
        model = smf.ols("dispersion ~ C(condition_label)", data=sub).fit(
            cov_type="cluster",
            cov_kwds={"groups": sub["Subject"]}
        )
        dispersion_stats_txt.append(f"\n=== terminal dispersion: {roi_name} ===\n")
        dispersion_stats_txt.append(str(model.summary()))
    except Exception as e:
        dispersion_stats_txt.append(f"\n=== terminal dispersion: {roi_name} ===\nFAILED: {e}\n")

with open(os.path.join(OUTPUT_DIR, "terminal_dispersion_stats.txt"), "w") as f:
    f.write("\n".join(dispersion_stats_txt))

# =========================================================
# reveal-to-terminal distance
# =========================================================
dist_rows = []
for (subj, game, cond), g in df_state.groupby(["Subject", "Game", "condition_label"]):
    g = g.sort_values("reveal_idx").copy()

    if g["has_acc_data"].iloc[0] == 1:
        gg = g.dropna(subset=["acc_PC1", "acc_PC2", "acc_PC3"])
        if len(gg) >= 1:
            X = gg[["acc_PC1", "acc_PC2", "acc_PC3"]].to_numpy(dtype=float)
            term = X[-1]
            d = np.sqrt(np.sum((X - term) ** 2, axis=1))
            for rr, dd in zip(gg["reveal_idx"].values, d):
                dist_rows.append({
                    "Subject": subj,
                    "Game": game,
                    "condition_label": cond,
                    "reveal_idx": rr,
                    "roi": "acc",
                    "distance_to_terminal": dd
                })

    if g["has_vmpfc_data"].iloc[0] == 1:
        gg = g.dropna(subset=["vmpfc_PC1", "vmpfc_PC2", "vmpfc_PC3"])
        if len(gg) >= 1:
            X = gg[["vmpfc_PC1", "vmpfc_PC2", "vmpfc_PC3"]].to_numpy(dtype=float)
            term = X[-1]
            d = np.sqrt(np.sum((X - term) ** 2, axis=1))
            for rr, dd in zip(gg["reveal_idx"].values, d):
                dist_rows.append({
                    "Subject": subj,
                    "Game": game,
                    "condition_label": cond,
                    "reveal_idx": rr,
                    "roi": "vmpfc",
                    "distance_to_terminal": dd
                })

    if g["has_acc_data"].iloc[0] == 1 and g["has_vmpfc_data"].iloc[0] == 1:
        gg = g.dropna(subset=["acc_PC1", "acc_PC2", "acc_PC3", "vmpfc_PC1", "vmpfc_PC2", "vmpfc_PC3"])
        if len(gg) >= 1:
            X = gg[["acc_PC1", "acc_PC2", "acc_PC3", "vmpfc_PC1", "vmpfc_PC2", "vmpfc_PC3"]].to_numpy(dtype=float)
            term = X[-1]
            d = np.sqrt(np.sum((X - term) ** 2, axis=1))
            for rr, dd in zip(gg["reveal_idx"].values, d):
                dist_rows.append({
                    "Subject": subj,
                    "Game": game,
                    "condition_label": cond,
                    "reveal_idx": rr,
                    "roi": "joint",
                    "distance_to_terminal": dd
                })

dist_to_terminal_df = pd.DataFrame(dist_rows)
dist_to_terminal_df.to_csv(os.path.join(OUTPUT_DIR, "distance_to_terminal_by_reveal.csv"), index=False)

for roi_name in ["acc", "vmpfc", "joint"]:
    sub = dist_to_terminal_df[dist_to_terminal_df["roi"] == roi_name].copy()
    if len(sub) == 0:
        continue

    mean_df = (
        sub.groupby(["condition_label", "reveal_idx"])["distance_to_terminal"]
        .mean()
        .reset_index()
    )

    plt.figure(figsize=(7, 5))
    for cond in ["equal", "unequal"]:
        ss = mean_df[mean_df["condition_label"] == cond].sort_values("reveal_idx")
        if len(ss) == 0:
            continue
        plt.plot(ss["reveal_idx"], ss["distance_to_terminal"], marker="o", label=cond)

    plt.xlabel("Reveal index")
    plt.ylabel("Distance to terminal state")
    plt.title(f"{roi_name.upper()} convergence to terminal state")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"{roi_name}_distance_to_terminal.png"), dpi=200)
    plt.close()

# =========================================================
# ROI coupling / divergence
# 只在同时有 ACC 和 vmPFC 的样本上有意义
# =========================================================
roi_div_df = df_state[
    (df_state["has_acc_data"] == 1)
    & (df_state["has_vmpfc_data"] == 1)
    & df_state[["roi_pc1_diff", "roi_pc2_diff", "roi_pc3_diff"]].notna().all(axis=1)
].copy()

roi_div_summary = (
    roi_div_df.groupby(["condition_label", "reveal_idx"])[
        ["roi_pc1_diff", "roi_pc2_diff", "roi_pc3_diff"]
    ]
    .mean()
    .reset_index()
)
roi_div_summary.to_csv(os.path.join(OUTPUT_DIR, "roi_divergence_summary.csv"), index=False)

for diff_col in ["roi_pc1_diff", "roi_pc2_diff", "roi_pc3_diff"]:
    plt.figure(figsize=(7, 5))
    for cond in ["equal", "unequal"]:
        sub = roi_div_summary[roi_div_summary["condition_label"] == cond].sort_values("reveal_idx")
        if len(sub) == 0:
            continue
        plt.plot(sub["reveal_idx"], sub[diff_col], marker="o", label=cond)

    plt.xlabel("Reveal index")
    plt.ylabel(diff_col)
    plt.title(f"ACC-vmPFC divergence: {diff_col}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"{diff_col}_trajectory.png"), dpi=200)
    plt.close()

# =========================================================
# save full corrected state table
# =========================================================
df_state.to_csv(os.path.join(OUTPUT_DIR, "reveal_level_with_roi_state_scores.csv"), index=False)

# =========================================================
# save subject inclusion report
# =========================================================
subject_inclusion = pd.DataFrame({
    "Subject": sorted(df["Subject"].unique())
})
subject_inclusion["has_acc_data"] = subject_inclusion["Subject"].isin(acc_valid_subjects).astype(int)
subject_inclusion["has_vmpfc_data"] = subject_inclusion["Subject"].isin(vmpfc_valid_subjects).astype(int)
subject_inclusion.to_csv(os.path.join(OUTPUT_DIR, "roi_subject_inclusion.csv"), index=False)

# =========================================================
# concise text summary scaffold
# =========================================================
summary_txt = []
summary_txt.append("ROI-valid subject counts:")
summary_txt.append(f"ACC valid subjects: {len(acc_valid_subjects)}")
summary_txt.append(f"vmPFC valid subjects: {len(vmpfc_valid_subjects)}")

summary_txt.append("\nACC explained variance:")
summary_txt.append(str(acc_explained))

summary_txt.append("\nvmPFC explained variance:")
summary_txt.append(str(vmpfc_explained))

summary_txt.append("\n\nGeometry condition means:")
summary_txt.append(str(geom_summary))

summary_txt.append("\n\nTerminal decode results:")
summary_txt.append(str(terminal_results_df))

summary_txt.append("\n\nRepresentation emergence slopes:")
if len(emergence_stats_df) > 0:
    summary_txt.append(str(emergence_stats_df.sort_values(['target', 'roi']).head(50)))
else:
    summary_txt.append("No emergence stats available.")

with open(os.path.join(OUTPUT_DIR, "quick_summary.txt"), "w") as f:
    f.write("\n".join(summary_txt))

print("\nDone.")
print("Corrected outputs overwritten in:", OUTPUT_DIR)
print("ACC valid subjects:", len(acc_valid_subjects))
print("vmPFC valid subjects:", len(vmpfc_valid_subjects))

Loaded: (10362, 195)
After complete-game filter: (10362, 195)
ACC feature n = 40
vmPFC feature n = 40
Available targets: ['current_reward', 'best_deck_snapshot', 'value_gap_snapshot', 'imbalance_change', 'current_deck_was_least_sampled_before', 'current_deck_is_best_snapshot']
ACC valid subjects = 10
vmPFC valid subjects = 19

Done.
Corrected outputs overwritten in: /root/autodl-tmp/reveal_roi_state_space_results_1/
ACC valid subjects: 10
vmPFC valid subjects: 19


# ACC decoding

# new parameters

In [4]:
import os
import shutil
import warnings
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from mpl_toolkits.mplot3d import Axes3D

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.model_selection import StratifiedKFold, KFold, cross_val_score
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore")

# =========================================================
# paths
# =========================================================
INPUT_CSV = "/root/autodl-tmp/reveal_neural_feature_tables/reveal_level_neural_feature_table.csv"
OUTPUT_DIR = "/root/autodl-tmp/reveal_roi_state_space_results/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 备份旧结果（可选，但建议保留）
BACKUP_DIR = os.path.join(OUTPUT_DIR, "_backup_old_wrong_outputs")
if not os.path.exists(BACKUP_DIR):
    os.makedirs(BACKUP_DIR, exist_ok=True)
    for fn in os.listdir(OUTPUT_DIR):
        src = os.path.join(OUTPUT_DIR, fn)
        dst = os.path.join(BACKUP_DIR, fn)
        if os.path.isfile(src):
            try:
                shutil.copy2(src, dst)
            except Exception:
                pass

# =========================================================
# Nature-style publication configuration
# =========================================================
def set_publication_style():
    """Nature 子刊风格配置：无衬线字体、可编辑矢量输出、合适字号与线宽"""
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
        "font.size": 7,
        "axes.labelsize": 7,
        "axes.titlesize": 7,
        "xtick.labelsize": 6,
        "ytick.labelsize": 6,
        "legend.fontsize": 6,
        "axes.linewidth": 0.8,
        "xtick.major.width": 0.8,
        "ytick.major.width": 0.8,
        "xtick.major.size": 3,
        "ytick.major.size": 3,
        "lines.linewidth": 1.2,
        "figure.dpi": 150,
        "savefig.dpi": 600,
        "savefig.bbox": "tight",
        "savefig.pad_inches": 0.02,
        "figure.facecolor": "white",
        "axes.facecolor": "white",
    })

set_publication_style()

# 色盲友好、克制配色（Nature常用风格）
COND_COLORS = {
    "equal": "#3C5488",    # 深蓝
    "unequal": "#B24745",  # 柔和红
}

COND_MARKERS = {
    "equal": "o",
    "unequal": "s",
}

REVEAL_LABEL_KW = dict(
    fontsize=5.5,
    ha="center",
    va="center",
    bbox=dict(boxstyle="round,pad=0.16", fc="white", ec="none", alpha=0.9)
)

def mm_to_inch(mm):
    return mm / 25.4

FIG_W_SINGLE = mm_to_inch(89)   # Nature 单栏常用宽度
FIG_W_DOUBLE = mm_to_inch(183)

def save_figure_multi(fig, outpath_base):
    """同时保存矢量（PDF/SVG）和高分辨率PNG，符合Nature投稿要求"""
    fig.savefig(outpath_base + ".pdf", transparent=True)
    fig.savefig(outpath_base + ".svg", transparent=True)
    fig.savefig(outpath_base + ".png", dpi=600, transparent=True)

# =========================================================
# load
# =========================================================
df = pd.read_csv(INPUT_CSV)
print("Loaded:", df.shape)

# 只保留完整 6-reveal game
df = df[df["game_complete_6"] == True].copy()
df = df.sort_values(["Subject", "Game", "reveal_idx"]).reset_index(drop=True)
print("After complete-game filter:", df.shape)

# =========================================================
# helper: pick feature columns by ROI
# =========================================================
def pick_roi_features(columns, roi_name):
    feats = []
    for c in columns:
        if not c.startswith(f"{roi_name}_"):
            continue
        if (
            "_bin" in c
            or c.endswith("_full_mean")
            or c.endswith("_full_auc")
            or c.endswith("_channel_sd_full")
        ):
            feats.append(c)
    return feats

acc_cols = pick_roi_features(df.columns, "acc")
vmpfc_cols = pick_roi_features(df.columns, "vmpfc")

# 去掉全空列
acc_cols = [c for c in acc_cols if df[c].notna().sum() > 0]
vmpfc_cols = [c for c in vmpfc_cols if df[c].notna().sum() > 0]

print("ACC feature n =", len(acc_cols))
print("vmPFC feature n =", len(vmpfc_cols))

# =========================================================
# sanity check
# =========================================================
required_cols = ["Subject", "Game", "condition_label", "reveal_idx"]
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

candidate_targets = [
    "current_reward",
    "best_deck_snapshot",
    "value_gap_snapshot",
    "sampling_imbalance_after",
    "sampling_imbalance_before",
    "imbalance_change",
    "current_deck_was_least_sampled_before",
    "current_deck_is_best_snapshot",
]

available_targets = [c for c in candidate_targets if c in df.columns]
print("Available targets:", available_targets)

# =========================================================
# helper: identify ROI-valid subjects / rows
# =========================================================
def get_valid_subjects_for_roi(df_in, feature_cols):
    if len(feature_cols) == 0:
        return []

    subj_valid = df_in.groupby("Subject")[feature_cols].apply(
        lambda x: x.notna().any().any()
    )
    valid_subjects = subj_valid[subj_valid].index.tolist()
    return valid_subjects

def get_valid_row_mask(df_in, feature_cols):
    if len(feature_cols) == 0:
        return pd.Series(False, index=df_in.index)

    return df_in[feature_cols].notna().any(axis=1)

acc_valid_subjects = get_valid_subjects_for_roi(df, acc_cols)
vmpfc_valid_subjects = get_valid_subjects_for_roi(df, vmpfc_cols)

print("ACC valid subjects =", len(acc_valid_subjects))
print("vmPFC valid subjects =", len(vmpfc_valid_subjects))

df["has_acc_data"] = df["Subject"].isin(acc_valid_subjects).astype(int)
df["has_vmpfc_data"] = df["Subject"].isin(vmpfc_valid_subjects).astype(int)

# =========================================================
# helper: preprocessing
# =========================================================
def preprocess_matrix(X):
    imputer = SimpleImputer(strategy="median")
    X_imp = imputer.fit_transform(X)

    scaler = StandardScaler()
    X_std = scaler.fit_transform(X_imp)
    return X_std, imputer, scaler

# =========================================================
# helper: centering
# =========================================================
def center_features(df_in, feature_cols, mode="subject_centered"):
    X = df_in[feature_cols].copy()

    if mode == "raw":
        return X

    out = X.copy()

    if mode == "subject_centered":
        subj_means = df_in.groupby("Subject")[feature_cols].transform("mean")
        out = X - subj_means

    elif mode == "subject_condition_centered":
        sc_means = df_in.groupby(["Subject", "condition_label"])[feature_cols].transform("mean")
        out = X - sc_means

    else:
        raise ValueError(f"Unknown center mode: {mode}")

    return out

# =========================================================
# helper: fit PCA for one ROI
# =========================================================
def run_roi_pca(df_in, roi_name, feature_cols, center_mode="subject_centered", n_components=3):
    if len(feature_cols) == 0:
        raise ValueError(f"No features for ROI={roi_name}")

    valid_row_mask = get_valid_row_mask(df_in, feature_cols)
    df_roi = df_in.loc[valid_row_mask].copy()

    if len(df_roi) == 0:
        raise ValueError(f"No valid rows for ROI={roi_name}")

    X_centered = center_features(df_roi, feature_cols, mode=center_mode)
    X_std, imputer, scaler = preprocess_matrix(X_centered.to_numpy(dtype=float))

    pca = PCA(n_components=n_components)
    X_pca = pca.fit_transform(X_std)

    key_cols = ["Subject", "Game", "condition_label", "reveal_idx"]
    if "Condition" in df_roi.columns:
        key_cols.append("Condition")

    score_df = df_roi[key_cols].copy().reset_index(drop=True)
    for i in range(n_components):
        score_df[f"{roi_name}_PC{i+1}"] = X_pca[:, i]

    loadings = pd.DataFrame(
        pca.components_.T,
        index=feature_cols,
        columns=[f"{roi_name}_PC{i+1}" for i in range(n_components)]
    )

    explained = pd.Series(
        pca.explained_variance_ratio_,
        index=[f"{roi_name}_PC{i+1}" for i in range(n_components)],
        name="explained_variance_ratio"
    )

    return score_df, loadings, explained, pca, imputer, scaler, df_roi

# =========================================================
# run ROI-specific PCA
# =========================================================
acc_scores, acc_loadings, acc_explained, acc_pca, _, _, df_acc = run_roi_pca(
    df[df["has_acc_data"] == 1].copy(),
    "acc",
    acc_cols,
    center_mode="subject_centered",
    n_components=3
)

vmpfc_scores, vmpfc_loadings, vmpfc_explained, vmpfc_pca, _, _, df_vmpfc = run_roi_pca(
    df[df["has_vmpfc_data"] == 1].copy(),
    "vmpfc",
    vmpfc_cols,
    center_mode="subject_centered",
    n_components=3
)

# 保存 PCA 输出
acc_loadings.to_csv(os.path.join(OUTPUT_DIR, "acc_pca_loadings.csv"))
vmpfc_loadings.to_csv(os.path.join(OUTPUT_DIR, "vmpfc_pca_loadings.csv"))
acc_explained.to_csv(os.path.join(OUTPUT_DIR, "acc_explained_variance.csv"), header=True)
vmpfc_explained.to_csv(os.path.join(OUTPUT_DIR, "vmpfc_explained_variance.csv"), header=True)

# =========================================================
# merge back safely
# =========================================================
df_state = df.copy()

merge_keys = ["Subject", "Game", "condition_label", "reveal_idx"]
if "Condition" in df_state.columns and "Condition" in acc_scores.columns and "Condition" in vmpfc_scores.columns:
    merge_keys = ["Subject", "Game", "Condition", "condition_label", "reveal_idx"]

df_state = df_state.merge(
    acc_scores[merge_keys + ["acc_PC1", "acc_PC2", "acc_PC3"]],
    on=merge_keys,
    how="left"
)

df_state = df_state.merge(
    vmpfc_scores[merge_keys + ["vmpfc_PC1", "vmpfc_PC2", "vmpfc_PC3"]],
    on=merge_keys,
    how="left"
)

# ROI divergence / coupling proxy
for i in [1, 2, 3]:
    df_state[f"roi_pc{i}_diff"] = df_state[f"acc_PC{i}"] - df_state[f"vmpfc_PC{i}"]

# =========================================================
# helper: subject/group summaries for trajectories
# =========================================================
def summarize_roi_trajectory(df_in, roi_prefix, valid_flag_col):
    sub0 = df_in[df_in[valid_flag_col] == 1].copy()
    sub0 = sub0.dropna(subset=[f"{roi_prefix}_PC1", f"{roi_prefix}_PC2", f"{roi_prefix}_PC3"])

    if len(sub0) == 0:
        return None, None

    # subject-level mean trajectory
    subj_traj = (
        sub0.groupby(["Subject", "condition_label", "reveal_idx"])[
            [f"{roi_prefix}_PC1", f"{roi_prefix}_PC2", f"{roi_prefix}_PC3"]
        ]
        .mean()
        .reset_index()
    )

    # grand mean + SEM across subjects
    grand = (
        subj_traj.groupby(["condition_label", "reveal_idx"])
        .agg(
            PC1_mean=(f"{roi_prefix}_PC1", "mean"),
            PC1_sem=(f"{roi_prefix}_PC1", lambda x: np.std(x, ddof=1) / np.sqrt(len(x)) if len(x) > 1 else np.nan),
            PC2_mean=(f"{roi_prefix}_PC2", "mean"),
            PC2_sem=(f"{roi_prefix}_PC2", lambda x: np.std(x, ddof=1) / np.sqrt(len(x)) if len(x) > 1 else np.nan),
            PC3_mean=(f"{roi_prefix}_PC3", "mean"),
            PC3_sem=(f"{roi_prefix}_PC3", lambda x: np.std(x, ddof=1) / np.sqrt(len(x)) if len(x) > 1 else np.nan),
            n_subjects=("Subject", "nunique"),
        )
        .reset_index()
    )

    return subj_traj, grand

# =========================================================
# improved 2D trajectory (Nature-style)
# =========================================================
def plot_roi_trajectory_2d(
    df_in,
    roi_prefix,
    explained_series,
    outpath_base,
    valid_flag_col,
    show_individual=True,
):
    subj_traj, grand = summarize_roi_trajectory(df_in, roi_prefix, valid_flag_col)
    if subj_traj is None:
        return

    fig, ax = plt.subplots(figsize=(FIG_W_SINGLE, FIG_W_SINGLE * 0.78))

    # optional: light individual trajectories
    if show_individual:
        for cond in ["equal", "unequal"]:
            ss = subj_traj[subj_traj["condition_label"] == cond].copy()
            for subj, g in ss.groupby("Subject"):
                g = g.sort_values("reveal_idx")
                ax.plot(
                    g[f"{roi_prefix}_PC1"],
                    g[f"{roi_prefix}_PC2"],
                    color=COND_COLORS[cond],
                    alpha=0.12,
                    linewidth=0.8,
                    zorder=1,
                )

    # grand mean trajectories
    for cond in ["equal", "unequal"]:
        sub = grand[grand["condition_label"] == cond].sort_values("reveal_idx")
        if len(sub) == 0:
            continue

        x = sub["PC1_mean"].to_numpy()
        y = sub["PC2_mean"].to_numpy()
        xerr = sub["PC1_sem"].to_numpy()
        yerr = sub["PC2_sem"].to_numpy()

        # SEM cross-hairs
        ax.errorbar(
            x, y,
            xerr=xerr, yerr=yerr,
            fmt="none",
            ecolor=COND_COLORS[cond],
            elinewidth=0.9,
            capsize=0,
            alpha=0.55,
            zorder=2,
        )

        # main line + markers
        ax.plot(
            x, y,
            color=COND_COLORS[cond],
            marker=COND_MARKERS[cond],
            markersize=4.2,
            linewidth=1.6,
            label=cond,
            zorder=3,
        )

        # directional arrows
        for i in range(len(sub) - 1):
            ax.annotate(
                "",
                xy=(x[i+1], y[i+1]),
                xytext=(x[i], y[i]),
                arrowprops=dict(
                    arrowstyle="-|>",
                    color=COND_COLORS[cond],
                    lw=1.0,
                    shrinkA=2,
                    shrinkB=2,
                    alpha=0.9,
                    mutation_scale=8,
                ),
                zorder=3,
            )

        # reveal labels
        for _, row in sub.iterrows():
            ax.text(
                row["PC1_mean"],
                row["PC2_mean"],
                str(int(row["reveal_idx"])),
                color="black",
                zorder=4,
                **REVEAL_LABEL_KW
            )

    # origin reference
    ax.axhline(0, color="0.85", linewidth=0.8, zorder=0)
    ax.axvline(0, color="0.85", linewidth=0.8, zorder=0)

    # labels
    ax.set_xlabel(f"{roi_prefix.upper()} PC1 ({explained_series.iloc[0]*100:.1f}%)")
    ax.set_ylabel(f"{roi_prefix.upper()} PC2 ({explained_series.iloc[1]*100:.1f}%)")

    # clean spines
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # auto margins
    x_all = grand["PC1_mean"].to_numpy()
    y_all = grand["PC2_mean"].to_numpy()
    if len(x_all) > 0 and len(y_all) > 0:
        xpad = max(np.nanstd(x_all) * 0.35, 0.3)
        ypad = max(np.nanstd(y_all) * 0.35, 0.3)
        ax.set_xlim(np.nanmin(x_all) - xpad, np.nanmax(x_all) + xpad)
        ax.set_ylim(np.nanmin(y_all) - ypad, np.nanmax(y_all) + ypad)

    ax.legend(frameon=False, loc="best", handlelength=1.8)

    fig.tight_layout()
    save_figure_multi(fig, outpath_base)
    plt.close(fig)

# =========================================================
# improved 3D trajectory (suitable for supplementary)
# =========================================================
def plot_roi_trajectory_3d(
    df_in,
    roi_prefix,
    explained_series,
    outpath_base,
    valid_flag_col,
    show_individual=True,
    elev=22,
    azim=-58,
):
    subj_traj, grand = summarize_roi_trajectory(df_in, roi_prefix, valid_flag_col)
    if subj_traj is None:
        return

    fig = plt.figure(figsize=(FIG_W_SINGLE, FIG_W_SINGLE * 0.92))
    ax = fig.add_subplot(111, projection="3d")

    # optional subject trajectories
    if show_individual:
        for cond in ["equal", "unequal"]:
            ss = subj_traj[subj_traj["condition_label"] == cond].copy()
            for subj, g in ss.groupby("Subject"):
                g = g.sort_values("reveal_idx")
                ax.plot(
                    g[f"{roi_prefix}_PC1"],
                    g[f"{roi_prefix}_PC2"],
                    g[f"{roi_prefix}_PC3"],
                    color=COND_COLORS[cond],
                    alpha=0.10,
                    linewidth=0.7,
                    zorder=1,
                )

    # grand mean trajectories
    for cond in ["equal", "unequal"]:
        sub = grand[grand["condition_label"] == cond].sort_values("reveal_idx")
        if len(sub) == 0:
            continue

        x = sub["PC1_mean"].to_numpy()
        y = sub["PC2_mean"].to_numpy()
        z = sub["PC3_mean"].to_numpy()

        ax.plot(
            x, y, z,
            color=COND_COLORS[cond],
            linewidth=1.8,
            zorder=3,
        )
        ax.scatter(
            x, y, z,
            color=COND_COLORS[cond],
            s=22,
            marker=COND_MARKERS[cond],
            depthshade=False,
            zorder=4,
        )

        for _, row in sub.iterrows():
            ax.text(
                row["PC1_mean"],
                row["PC2_mean"],
                row["PC3_mean"],
                str(int(row["reveal_idx"])),
                fontsize=5.5,
                color="black",
                zorder=5,
            )

    # labels
    ax.set_xlabel(f"PC1 ({explained_series.iloc[0]*100:.1f}%)", labelpad=3)
    ax.set_ylabel(f"PC2 ({explained_series.iloc[1]*100:.1f}%)", labelpad=3)
    ax.set_zlabel(f"PC3 ({explained_series.iloc[2]*100:.1f}%)", labelpad=3)

    # view angle
    ax.view_init(elev=elev, azim=azim)

    # clean 3D panes
    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False
    ax.xaxis.pane.set_edgecolor("white")
    ax.yaxis.pane.set_edgecolor("white")
    ax.zaxis.pane.set_edgecolor("white")

    # grid
    try:
        for axis in [ax.xaxis, ax.yaxis, ax.zaxis]:
            axis._axinfo["grid"]["color"] = (0.90, 0.90, 0.90, 1)
            axis._axinfo["grid"]["linewidth"] = 0.6
    except Exception:
        pass

    # legend
    handles = [
        Line2D([0], [0], color=COND_COLORS["equal"], marker=COND_MARKERS["equal"],
               lw=1.6, markersize=4, label="equal"),
        Line2D([0], [0], color=COND_COLORS["unequal"], marker=COND_MARKERS["unequal"],
               lw=1.6, markersize=4, label="unequal"),
    ]
    ax.legend(handles=handles, frameon=False, loc="upper left", bbox_to_anchor=(0.02, 0.98))

    fig.tight_layout()
    save_figure_multi(fig, outpath_base)
    plt.close(fig)

# =========================================================
# 替换原来的2D绘图，并新增3D版本
# =========================================================
plot_roi_trajectory_2d(
    df_state,
    "acc",
    acc_explained,
    os.path.join(OUTPUT_DIR, "acc_subject_centered_trajectory"),
    valid_flag_col="has_acc_data",
    show_individual=True
)

plot_roi_trajectory_2d(
    df_state,
    "vmpfc",
    vmpfc_explained,
    os.path.join(OUTPUT_DIR, "vmpfc_subject_centered_trajectory"),
    valid_flag_col="has_vmpfc_data",
    show_individual=True
)

plot_roi_trajectory_3d(
    df_state,
    "acc",
    acc_explained,
    os.path.join(OUTPUT_DIR, "acc_subject_centered_trajectory_3d"),
    valid_flag_col="has_acc_data",
    show_individual=True,
)

plot_roi_trajectory_3d(
    df_state,
    "vmpfc",
    vmpfc_explained,
    os.path.join(OUTPUT_DIR, "vmpfc_subject_centered_trajectory_3d"),
    valid_flag_col="has_vmpfc_data",
    show_individual=True,
)

# =========================================================
# 以下为原代码剩余部分（统计、几何、表征等），保持不变
# =========================================================
def compute_geometry_metrics_for_group(g, pc_cols, prefix):
    g = g.sort_values("reveal_idx").copy()
    g = g.dropna(subset=pc_cols)

    X = g[pc_cols].to_numpy(dtype=float)
    if X.shape[0] < 2:
        return None

    step_vecs = np.diff(X, axis=0)
    step_sizes = np.sqrt(np.sum(step_vecs ** 2, axis=1))

    traj_length = np.sum(step_sizes)
    mean_step = np.mean(step_sizes)
    max_step = np.max(step_sizes)

    displacement = np.sqrt(np.sum((X[-1] - X[0]) ** 2))
    efficiency = displacement / traj_length if traj_length > 1e-8 else np.nan

    turning_angles = []
    for i in range(len(step_vecs) - 1):
        a = step_vecs[i]
        b = step_vecs[i + 1]
        na = np.linalg.norm(a)
        nb = np.linalg.norm(b)
        if na < 1e-8 or nb < 1e-8:
            turning_angles.append(np.nan)
            continue
        cosang = np.dot(a, b) / (na * nb)
        cosang = np.clip(cosang, -1, 1)
        ang = np.arccos(cosang)
        turning_angles.append(ang)

    mean_turn_angle = np.nanmean(turning_angles) if len(turning_angles) > 0 else np.nan

    terminal = X[-1]
    dist_to_terminal = np.sqrt(np.sum((X - terminal) ** 2, axis=1))
    mean_dist_to_terminal = np.mean(dist_to_terminal[:-1]) if len(dist_to_terminal) > 1 else np.nan

    out = {
        f"{prefix}_traj_length": traj_length,
        f"{prefix}_mean_step": mean_step,
        f"{prefix}_max_step": max_step,
        f"{prefix}_displacement": displacement,
        f"{prefix}_efficiency": efficiency,
        f"{prefix}_mean_turn_angle": mean_turn_angle,
        f"{prefix}_mean_dist_to_terminal": mean_dist_to_terminal,
    }
    return out

geometry_rows = []
for (subj, game, cond), g in df_state.groupby(["Subject", "Game", "condition_label"]):
    row = {
        "Subject": subj,
        "Game": game,
        "condition_label": cond,
        "has_acc_data": int(g["has_acc_data"].iloc[0]),
        "has_vmpfc_data": int(g["has_vmpfc_data"].iloc[0]),
    }

    if row["has_acc_data"] == 1:
        acc_metrics = compute_geometry_metrics_for_group(
            g, ["acc_PC1", "acc_PC2", "acc_PC3"], "acc"
        )
        if acc_metrics is not None:
            row.update(acc_metrics)

    if row["has_vmpfc_data"] == 1:
        vmpfc_metrics = compute_geometry_metrics_for_group(
            g, ["vmpfc_PC1", "vmpfc_PC2", "vmpfc_PC3"], "vmpfc"
        )
        if vmpfc_metrics is not None:
            row.update(vmpfc_metrics)

    if row["has_acc_data"] == 1 and row["has_vmpfc_data"] == 1:
        joint_metrics = compute_geometry_metrics_for_group(
            g,
            ["acc_PC1", "acc_PC2", "acc_PC3", "vmpfc_PC1", "vmpfc_PC2", "vmpfc_PC3"],
            "joint"
        )
        if joint_metrics is not None:
            row.update(joint_metrics)

    geometry_rows.append(row)

geometry_df = pd.DataFrame(geometry_rows)
geometry_df.to_csv(os.path.join(OUTPUT_DIR, "trajectory_geometry_metrics.csv"), index=False)

geom_summary = geometry_df.groupby("condition_label").mean(numeric_only=True)
geom_summary.to_csv(os.path.join(OUTPUT_DIR, "trajectory_geometry_condition_means.csv"))

# （后续几何统计、表征涌现、terminal分析、距离计算等部分保持不变，为节省篇幅此处省略直接复制原代码剩余内容）

# =========================================================
# save full corrected state table
# =========================================================
df_state.to_csv(os.path.join(OUTPUT_DIR, "reveal_level_with_roi_state_scores.csv"), index=False)

# =========================================================
# save subject inclusion report
# =========================================================
subject_inclusion = pd.DataFrame({
    "Subject": sorted(df["Subject"].unique())
})
subject_inclusion["has_acc_data"] = subject_inclusion["Subject"].isin(acc_valid_subjects).astype(int)
subject_inclusion["has_vmpfc_data"] = subject_inclusion["Subject"].isin(vmpfc_valid_subjects).astype(int)
subject_inclusion.to_csv(os.path.join(OUTPUT_DIR, "roi_subject_inclusion.csv"), index=False)

print("\nDone.")
print("优化后的2D与3D trajectory图已生成（PDF/SVG/PNG），保存在：", OUTPUT_DIR)
print("ACC valid subjects:", len(acc_valid_subjects))
print("vmPFC valid subjects:", len(vmpfc_valid_subjects))

Loaded: (10362, 195)
After complete-game filter: (10362, 195)
ACC feature n = 40
vmPFC feature n = 40
Available targets: ['current_reward', 'best_deck_snapshot', 'value_gap_snapshot', 'imbalance_change', 'current_deck_was_least_sampled_before', 'current_deck_is_best_snapshot']
ACC valid subjects = 10
vmPFC valid subjects = 19

Done.
优化后的2D与3D trajectory图已生成（PDF/SVG/PNG），保存在： /root/autodl-tmp/reveal_roi_state_space_results/
ACC valid subjects: 10
vmPFC valid subjects: 19


# bestdeck change focus

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from mpl_toolkits.mplot3d import Axes3D
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

warnings.filterwarnings("ignore")

# =========================================================
# paths
# =========================================================
INPUT_CSV = "/root/acc_normative_full_analysis/df_state_with_normative_metrics.csv"
OUTPUT_DIR = "/root/acc_bestdeck_change_main_analysis/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================================================
# load
# =========================================================
df = pd.read_csv(INPUT_CSV)
print("Loaded:", df.shape)

required_cols = [
    "Subject", "Game", "reveal_idx", "condition_label",
    "acc_PC1", "acc_PC2", "acc_PC3",
    "bestdeck_changed_after_reveal"
]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = df.sort_values(["Subject", "Game", "reveal_idx"]).reset_index(drop=True)
df["bestdeck_changed_after_reveal"] = pd.to_numeric(df["bestdeck_changed_after_reveal"], errors="coerce")
df = df.dropna(subset=["Subject", "Game", "reveal_idx", "acc_PC1", "acc_PC2", "acc_PC3", "bestdeck_changed_after_reveal"]).copy()

# 只保留完整 game（如果你表里有这个字段）
if "game_complete_6" in df.columns:
    df = df[df["game_complete_6"] == True].copy()

# =========================================================
# helpers
# =========================================================
def euclid(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    return float(np.sqrt(np.sum((a - b) ** 2)))

def bootstrap_mean_ci(x, n_boot=2000, alpha=0.05, random_state=42):
    x = np.array(pd.Series(x).dropna())
    if len(x) == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(random_state)
    boots = []
    for _ in range(n_boot):
        samp = rng.choice(x, size=len(x), replace=True)
        boots.append(np.mean(samp))
    lo = np.percentile(boots, 100 * (alpha / 2))
    hi = np.percentile(boots, 100 * (1 - alpha / 2))
    return np.mean(x), lo, hi

def cv_auc_score(X, y, n_splits=5):
    y = np.asarray(y).astype(int)
    if len(np.unique(y)) < 2:
        return np.nan
    binc = np.bincount(y)
    if len(binc) < 2:
        return np.nan
    min_class = np.min(binc)
    ncv = min(n_splits, min_class)
    if ncv < 2:
        return np.nan
    clf = LogisticRegression(max_iter=2000)
    cv = StratifiedKFold(n_splits=ncv, shuffle=True, random_state=42)
    scores = cross_val_score(clf, X, y, cv=cv, scoring="roc_auc")
    return np.mean(scores)

def permutation_auc(X, y, n_perm=1000, random_state=42):
    obs = cv_auc_score(X, y)
    if np.isnan(obs):
        return np.nan, np.nan
    rng = np.random.default_rng(random_state)
    perm_scores = []
    for _ in range(n_perm):
        yp = rng.permutation(y)
        s = cv_auc_score(X, yp)
        if not np.isnan(s):
            perm_scores.append(s)
    perm_scores = np.asarray(perm_scores, dtype=float)
    if len(perm_scores) == 0:
        return obs, np.nan
    p = (np.sum(perm_scores >= obs) + 1) / (len(perm_scores) + 1)
    return obs, p
# Nature-style 配置（放在 helpers 部分）
def set_nature_style():
    plt.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
        "font.size": 7,
        "axes.labelsize": 7.5,
        "axes.titlesize": 8.8,
        "xtick.labelsize": 6.5,
        "ytick.labelsize": 6.5,
        "legend.fontsize": 6.8,
        "axes.linewidth": 0.8,
        "lines.linewidth": 1.6,
        "figure.dpi": 150,
        "savefig.dpi": 600,
        "savefig.bbox": "tight",
        "savefig.pad_inches": 0.02,
        "figure.facecolor": "white",
        "axes.facecolor": "white",
    })

FIG_W = 183 / 25.4   # Nature 双栏宽度
# =========================================================
# 1) compute ACC state update metrics
# step t = distance(state_t - state_{t-1})
# this step is aligned to the current reveal row t
# =========================================================
df[["acc_PC1_prev", "acc_PC2_prev", "acc_PC3_prev"]] = (
    df.groupby(["Subject", "Game"])[["acc_PC1", "acc_PC2", "acc_PC3"]].shift(1)
)

df["acc_state_update"] = np.sqrt(
    (df["acc_PC1"] - df["acc_PC1_prev"]) ** 2 +
    (df["acc_PC2"] - df["acc_PC2_prev"]) ** 2 +
    (df["acc_PC3"] - df["acc_PC3_prev"]) ** 2
)

df["acc_update_PC1_abs"] = np.abs(df["acc_PC1"] - df["acc_PC1_prev"])
df["acc_update_PC2_abs"] = np.abs(df["acc_PC2"] - df["acc_PC2_prev"])
df["acc_update_PC3_abs"] = np.abs(df["acc_PC3"] - df["acc_PC3_prev"])

# reveal 1 没有前一步
df_update = df[df["reveal_idx"] >= 2].copy()

# =========================================================
# 2) main model: state update ~ bestdeck_changed_after_reveal
# =========================================================
main_txt = []

# pooled clustered OLS
try:
    m1 = smf.ols(
        "acc_state_update ~ bestdeck_changed_after_reveal + C(reveal_idx) + C(condition_label)",
        data=df_update
    ).fit(cov_type="cluster", cov_kwds={"groups": df_update["Subject"]})
    main_txt.append("=== Main model: acc_state_update ~ bestdeck_changed_after_reveal + reveal + condition ===\n")
    main_txt.append(str(m1.summary()))
except Exception as e:
    main_txt.append(f"Main model failed: {e}")

# interaction with condition
try:
    m2 = smf.ols(
        "acc_state_update ~ bestdeck_changed_after_reveal * C(condition_label) + C(reveal_idx)",
        data=df_update
    ).fit(cov_type="cluster", cov_kwds={"groups": df_update["Subject"]})
    main_txt.append("\n\n=== Interaction model: acc_state_update ~ bestdeck_changed_after_reveal * condition + reveal ===\n")
    main_txt.append(str(m2.summary()))
except Exception as e:
    main_txt.append(f"Interaction model failed: {e}")

# reveal-specific models
for rv in sorted(df_update["reveal_idx"].unique()):
    sub = df_update[df_update["reveal_idx"] == rv].copy()
    if sub["bestdeck_changed_after_reveal"].nunique() < 2:
        continue
    try:
        mm = smf.ols(
            "acc_state_update ~ bestdeck_changed_after_reveal + C(condition_label)",
            data=sub
        ).fit(cov_type="cluster", cov_kwds={"groups": sub["Subject"]})
        main_txt.append(f"\n\n=== Reveal {rv}: acc_state_update ~ bestdeck_changed_after_reveal + condition ===\n")
        main_txt.append(str(mm.summary()))
    except Exception as e:
        main_txt.append(f"\nReveal {rv} model failed: {e}")

with open(os.path.join(OUTPUT_DIR, "main_models.txt"), "w") as f:
    f.write("\n".join(main_txt))

# =========================================================
# 3) subject-level summary analysis
# 更稳一点：每个被试每个reveal，switch vs no-switch 的平均 update
# =========================================================
subj_rows = []
for (subj, rv), g in df_update.groupby(["Subject", "reveal_idx"]):
    for sw in [0, 1]:
        gg = g[g["bestdeck_changed_after_reveal"] == sw]
        if len(gg) == 0:
            continue
        subj_rows.append({
            "Subject": subj,
            "reveal_idx": rv,
            "switch": sw,
            "mean_acc_state_update": gg["acc_state_update"].mean(),
            "n_trials": len(gg)
        })

subj_df = pd.DataFrame(subj_rows)
subj_df.to_csv(os.path.join(OUTPUT_DIR, "subject_reveal_switch_update_summary.csv"), index=False)

subj_txt = []
if len(subj_df) > 0:
    try:
        smod = smf.ols(
            "mean_acc_state_update ~ switch + C(reveal_idx)",
            data=subj_df
        ).fit(cov_type="cluster", cov_kwds={"groups": subj_df["Subject"]})
        subj_txt.append("=== Subject-level model: mean update ~ switch + reveal ===\n")
        subj_txt.append(str(smod.summary()))
    except Exception as e:
        subj_txt.append(f"Subject-level model failed: {e}")

    for rv, g in subj_df.groupby("reveal_idx"):
        g0 = g[g["switch"] == 0][["Subject", "mean_acc_state_update"]].rename(columns={"mean_acc_state_update": "u0"})
        g1 = g[g["switch"] == 1][["Subject", "mean_acc_state_update"]].rename(columns={"mean_acc_state_update": "u1"})
        mm = g0.merge(g1, on="Subject", how="inner")
        if len(mm) > 2:
            mm["delta_switch_minus_noswitch"] = mm["u1"] - mm["u0"]
            meanv, lo, hi = bootstrap_mean_ci(mm["delta_switch_minus_noswitch"].values)
            subj_txt.append(
                f"\nReveal {rv}: switch - no-switch mean delta = {meanv:.4f}, 95% bootstrap CI = [{lo:.4f}, {hi:.4f}], n={len(mm)}"
            )

with open(os.path.join(OUTPUT_DIR, "subject_level_stats.txt"), "w") as f:
    f.write("\n".join(subj_txt))

# =========================================================
# 4) decode bestdeck_changed_after_reveal from ACC state
# overall + by reveal + by condition
# =========================================================
decode_rows = []

# overall
sub_all = df.copy()
sub_all = sub_all.dropna(subset=["acc_PC1", "acc_PC2", "acc_PC3", "bestdeck_changed_after_reveal"])
X = sub_all[["acc_PC1", "acc_PC2", "acc_PC3"]].to_numpy(dtype=float)
y = sub_all["bestdeck_changed_after_reveal"].astype(int).to_numpy()
auc, p = permutation_auc(X, y, n_perm=2000, random_state=42)
decode_rows.append({
    "scope": "overall",
    "condition_label": "all",
    "reveal_idx": -1,
    "auc": auc,
    "perm_p": p,
    "n_trials": len(sub_all),
    "n_switch0": int(np.sum(y == 0)),
    "n_switch1": int(np.sum(y == 1)),
})

# by reveal
for rv in sorted(df["reveal_idx"].unique()):
    sub = df[df["reveal_idx"] == rv].copy()
    sub = sub.dropna(subset=["acc_PC1", "acc_PC2", "acc_PC3", "bestdeck_changed_after_reveal"])
    y = sub["bestdeck_changed_after_reveal"].astype(int).to_numpy()
    X = sub[["acc_PC1", "acc_PC2", "acc_PC3"]].to_numpy(dtype=float)
    auc, p = permutation_auc(X, y, n_perm=1000, random_state=42)
    decode_rows.append({
        "scope": "by_reveal",
        "condition_label": "all",
        "reveal_idx": int(rv),
        "auc": auc,
        "perm_p": p,
        "n_trials": len(sub),
        "n_switch0": int(np.sum(y == 0)),
        "n_switch1": int(np.sum(y == 1)),
    })

# by reveal x condition
for cond in sorted(df["condition_label"].dropna().unique()):
    for rv in sorted(df["reveal_idx"].unique()):
        sub = df[(df["condition_label"] == cond) & (df["reveal_idx"] == rv)].copy()
        sub = sub.dropna(subset=["acc_PC1", "acc_PC2", "acc_PC3", "bestdeck_changed_after_reveal"])
        if len(sub) == 0:
            continue
        y = sub["bestdeck_changed_after_reveal"].astype(int).to_numpy()
        X = sub[["acc_PC1", "acc_PC2", "acc_PC3"]].to_numpy(dtype=float)
        auc, p = permutation_auc(X, y, n_perm=1000, random_state=42)
        decode_rows.append({
            "scope": "by_reveal_condition",
            "condition_label": cond,
            "reveal_idx": int(rv),
            "auc": auc,
            "perm_p": p,
            "n_trials": len(sub),
            "n_switch0": int(np.sum(y == 0)),
            "n_switch1": int(np.sum(y == 1)),
        })

decode_df = pd.DataFrame(decode_rows)
decode_df.to_csv(os.path.join(OUTPUT_DIR, "decode_bestdeck_changed_from_acc.csv"), index=False)

# =========================================================
# 5) event frequency analysis
# 看 switch 在各reveal/condition下发生率
# =========================================================
freq_df = (
    df.groupby(["condition_label", "reveal_idx"])["bestdeck_changed_after_reveal"]
    .agg(["mean", "sum", "count"])
    .reset_index()
    .rename(columns={"mean": "switch_rate", "sum": "n_switch", "count": "n_trials"})
)
freq_df.to_csv(os.path.join(OUTPUT_DIR, "bestdeck_change_frequency.csv"), index=False)

# =========================================================
# 6) Visualization (Nature-style, refined)
# =========================================================

set_nature_style()   # 使用之前trajectory图里定义的Nature风格函数（如果还没有，请把下面的set_nature_style复制进去）

def save_figure(fig, base_name):
    """同时保存投稿友好格式"""
    fig.savefig(os.path.join(OUTPUT_DIR, base_name + ".pdf"), transparent=True)
    fig.savefig(os.path.join(OUTPUT_DIR, base_name + ".svg"), transparent=True)
    fig.savefig(os.path.join(OUTPUT_DIR, base_name + ".png"), dpi=600, transparent=True)

print("\n开始生成精炼可视化...")

# A. Reveal-wise decode AUC
sub = decode_df[(decode_df["scope"] == "by_reveal") & (decode_df["condition_label"] == "all")].copy()
if len(sub) > 0:
    fig, ax = plt.subplots(figsize=(FIG_W * 0.65, FIG_W * 0.48))
    ax.plot(sub["reveal_idx"], sub["auc"], marker="o", color="#2980B9", linewidth=2.2, markersize=5.5)
    ax.axhline(0.5, linestyle="--", linewidth=1.2, color="gray", alpha=0.7)
    for _, r in sub.iterrows():
        if not np.isnan(r["perm_p"]):
            ax.text(r["reveal_idx"] + 0.08, r["auc"] + 0.012, f"p={r['perm_p']:.3f}", 
                   fontsize=6.5, color="#2C3E50")
    ax.set_xlabel("Reveal index")
    ax.set_ylabel("Decode AUC (best-deck change)")
    ax.set_title("ACC decodes belief reconfiguration\nafter reveal", pad=12, fontweight="bold")
    ax.set_xticks(range(1, 7))
    ax.set_ylim(0.4, 1.02)
    ax.grid(True, alpha=0.25, linestyle="--")
    ax.legend(["Observed AUC", "Chance level"], frameon=False, fontsize=6.8, loc="lower right")
    plt.tight_layout()
    save_figure(fig, "fig1_decode_auc_by_reveal")
    plt.close(fig)
    print("✓ 已保存: fig1_decode_auc_by_reveal.pdf")

# B. ACC state update by reveal and switch
upd_plot = (
    df_update.groupby(["reveal_idx", "bestdeck_changed_after_reveal"])["acc_state_update"]
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(FIG_W * 0.68, FIG_W * 0.50))
palette = {0: "#7F8C8E", 1: "#E74C3C"}
labels = {0: "No reconfiguration", 1: "Reconfiguration"}
for sw in [0, 1]:
    ss = upd_plot[upd_plot["bestdeck_changed_after_reveal"] == sw].sort_values("reveal_idx")
    if len(ss) > 0:
        ax.plot(ss["reveal_idx"], ss["acc_state_update"], marker="o", linewidth=2.1, 
                markersize=5.5, color=palette[sw], label=labels[sw])
ax.set_xlabel("Reveal index")
ax.set_ylabel("Mean ACC state update\n(Euclidean distance in PC space)")
ax.set_title("ACC state update magnitude\nby belief reconfiguration", pad=12, fontweight="bold")
ax.set_xticks(range(1, 7))
ax.grid(True, alpha=0.25, linestyle="--")
ax.legend(frameon=False, fontsize=6.8, loc="best")
plt.tight_layout()
save_figure(fig, "fig2_state_update_by_reveal_switch")
plt.close(fig)
print("✓ 已保存: fig2_state_update_by_reveal_switch.pdf")

# C. ACC state update by condition and switch (bar plot)
upd_cond = (
    df_update.groupby(["condition_label", "bestdeck_changed_after_reveal"])["acc_state_update"]
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(FIG_W * 0.62, FIG_W * 0.52))
conds = sorted(upd_cond["condition_label"].dropna().unique())
x = np.arange(len(conds))
width = 0.35
palette_bar = {"equal": "#1F7CC2", "unequal": "#FF7F2A"}

vals0 = []
vals1 = []
for c in conds:
    v0 = upd_cond[(upd_cond["condition_label"] == c) & (upd_cond["bestdeck_changed_after_reveal"] == 0)]["acc_state_update"].mean()
    v1 = upd_cond[(upd_cond["condition_label"] == c) & (upd_cond["bestdeck_changed_after_reveal"] == 1)]["acc_state_update"].mean()
    vals0.append(v0 if not np.isnan(v0) else 0)
    vals1.append(v1 if not np.isnan(v1) else 0)

ax.bar(x - width/2, vals0, width, label="No reconfiguration", color="#7F8C8E", edgecolor="black", linewidth=0.8)
ax.bar(x + width/2, vals1, width, label="Reconfiguration", color="#E74C3C", edgecolor="black", linewidth=0.8)

ax.set_xticks(x)
ax.set_xticklabels(conds, fontsize=7)
ax.set_ylabel("Mean ACC state update")
ax.set_title("ACC update by reward condition\nand belief reconfiguration", pad=12, fontweight="bold")
ax.grid(True, alpha=0.15, axis="y")
ax.legend(frameon=False, fontsize=6.8, loc="best")
plt.tight_layout()
save_figure(fig, "fig3_state_update_condition_switch")
plt.close(fig)
print("✓ 已保存: fig3_state_update_condition_switch.pdf")

# D. Belief-switch frequency by reveal and condition
fig, ax = plt.subplots(figsize=(FIG_W * 0.68, FIG_W * 0.50))
for cond in sorted(freq_df["condition_label"].dropna().unique()):
    ss = freq_df[freq_df["condition_label"] == cond].sort_values("reveal_idx")
    color = "#1F7CC2" if cond == "equal" else "#FF7F2A"
    ax.plot(ss["reveal_idx"], ss["switch_rate"], marker="o", linewidth=2.1, 
            markersize=5.5, label=cond.capitalize(), color=color)
ax.set_xlabel("Reveal index")
ax.set_ylabel("Belief reconfiguration rate")
ax.set_title("Belief-switch frequency across reveals", pad=12, fontweight="bold")
ax.set_xticks(range(1, 7))
ax.set_ylim(0, 1.02)
ax.grid(True, alpha=0.25, linestyle="--")
ax.legend(title="Condition", frameon=False, fontsize=6.8, loc="best")
plt.tight_layout()
save_figure(fig, "fig4_switch_rate_by_reveal_condition")
plt.close(fig)
print("✓ 已保存: fig4_switch_rate_by_reveal_condition.pdf")

# E. ACC PC1-PC2 state map at reveal 3 (switch vs no-switch)
sub3 = df[df["reveal_idx"] == 3].copy()
if len(sub3) > 0:
    fig, ax = plt.subplots(figsize=(FIG_W * 0.65, FIG_W * 0.52))
    palette_scatter = {0: "#7F8C8E", 1: "#E74C3C"}
    for sw, label in [(0, "No reconfiguration"), (1, "Reconfiguration")]:
        ss = sub3[sub3["bestdeck_changed_after_reveal"] == sw]
        if len(ss) > 0:
            ax.scatter(ss["acc_PC1"], ss["acc_PC2"], alpha=0.45, s=18, 
                      color=palette_scatter[sw], label=label, edgecolors="white", linewidth=0.4)
    ax.set_xlabel("ACC PC1")
    ax.set_ylabel("ACC PC2")
    ax.set_title("ACC neural state at reveal 3\nby belief reconfiguration", pad=12, fontweight="bold")
    ax.grid(True, alpha=0.15)
    ax.legend(frameon=False, fontsize=6.8, loc="best")
    plt.tight_layout()
    save_figure(fig, "fig5_acc_state_reveal3_switch")
    plt.close(fig)
    print("✓ 已保存: fig5_acc_state_reveal3_switch.pdf")

# F. Subject-level paired delta (switch - no-switch)
if len(subj_df) > 0:
    paired = []
    for rv, g in subj_df.groupby("reveal_idx"):
        g0 = g[g["switch"] == 0][["Subject", "mean_acc_state_update"]].rename(columns={"mean_acc_state_update": "u0"})
        g1 = g[g["switch"] == 1][["Subject", "mean_acc_state_update"]].rename(columns={"mean_acc_state_update": "u1"})
        mm = g0.merge(g1, on="Subject", how="inner")
        if len(mm) > 0:
            mm["delta"] = mm["u1"] - mm["u0"]
            mm["reveal_idx"] = rv
            paired.append(mm[["Subject", "reveal_idx", "delta"]])
    if len(paired) > 0:
        paired_df = pd.concat(paired, ignore_index=True)
        fig, ax = plt.subplots(figsize=(FIG_W * 0.65, FIG_W * 0.48))
        mean_df = paired_df.groupby("reveal_idx")["delta"].mean().reset_index()
        ax.plot(mean_df["reveal_idx"], mean_df["delta"], marker="o", color="#E74C3C", linewidth=2.2, markersize=5.5)
        ax.axhline(0, linestyle="--", linewidth=1.2, color="gray", alpha=0.7)
        ax.set_xlabel("Reveal index")
        ax.set_ylabel("Switch − No-switch\nACC state update (paired)")
        ax.set_title("Subject-level paired effect of\nbelief reconfiguration on ACC update", pad=12, fontweight="bold")
        ax.set_xticks(range(1, 7))
        ax.grid(True, alpha=0.25, linestyle="--")
        plt.tight_layout()
        save_figure(fig, "fig6_subject_paired_delta_by_reveal")
        plt.close(fig)
        print("✓ 已保存: fig6_subject_paired_delta_by_reveal.pdf")

print("\n所有可视化生成完成！")
print(f"输出目录: {OUTPUT_DIR}")
print("推荐投稿使用对应的 .pdf 文件（矢量格式，可直接编辑）。")

# =========================================================
# 7) concise summary
# =========================================================
summary = []

summary.append("=== Overall decode of bestdeck_changed_after_reveal from ACC ===")
summary.append(str(decode_df[decode_df["scope"] == "overall"]))
summary.append("")

summary.append("=== Reveal-wise decode ===")
summary.append(str(decode_df[decode_df["scope"] == "by_reveal"]))
summary.append("")

summary.append("=== Switch frequency by reveal and condition ===")
summary.append(str(freq_df))
summary.append("")

# peak AUC
tmp = decode_df[decode_df["scope"] == "by_reveal"].dropna(subset=["auc"]).copy()
if len(tmp) > 0:
    best = tmp.loc[tmp["auc"].idxmax()]
    summary.append(
        f"Peak reveal-wise decode: reveal {int(best['reveal_idx'])}, AUC={best['auc']:.4f}, p={best['perm_p']:.4f}"
    )

# update means
upd_sum = (
    df_update.groupby(["reveal_idx", "bestdeck_changed_after_reveal"])["acc_state_update"]
    .mean()
    .reset_index()
)
summary.append("")
summary.append("=== Mean ACC state update by reveal and switch ===")
summary.append(str(upd_sum))

with open(os.path.join(OUTPUT_DIR, "quick_summary.txt"), "w") as f:
    f.write("\n".join(summary))

print("Done.")
print("Saved to:", OUTPUT_DIR)

# vis for decoding fold line

In [8]:
import os
import matplotlib as mpl
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from mpl_toolkits.mplot3d import Axes3D
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

# =========================================================
# 6) Visualization (Nature-style, refined)
# =========================================================

set_nature_style()   # 使用之前trajectory图里定义的Nature风格函数（如果还没有，请把下面的set_nature_style复制进去）

def save_figure(fig, base_name):
    """同时保存投稿友好格式"""
   #fig.savefig(os.path.join(OUTPUT_DIR, base_name + ".pdf"), transparent=True)
    fig.savefig(os.path.join(OUTPUT_DIR, base_name + ".svg"), transparent=True)
    fig.savefig(os.path.join(OUTPUT_DIR, base_name + ".png"), dpi=600, transparent=True)

print("\n开始生成精炼可视化...")

# A. Reveal-wise decode AUC
sub = decode_df[(decode_df["scope"] == "by_reveal") & (decode_df["condition_label"] == "all")].copy()
if len(sub) > 0:
    fig, ax = plt.subplots(figsize=(FIG_W * 0.5, FIG_W * 0.48))
    ax.plot(sub["reveal_idx"], sub["auc"], marker="o", color="#67428F", linewidth=2.2, markersize=5.5)
    ax.axhline(0.5, linestyle="--", linewidth=1.2, color="gray", alpha=0.7)
    for _, r in sub.iterrows():
        if not np.isnan(r["perm_p"]):
            ax.text(r["reveal_idx"] + 0.08, r["auc"] + 0.012, f"p={r['perm_p']:.3f}", 
                   fontsize=6.5, color="#2C3E50")
    ax.set_xlabel("Reveal index")
    ax.set_ylabel("Decode AUC (best-deck change)")
    ax.set_title("ACC decodes belief reconfiguration\nafter reveal", pad=12, fontweight="bold")
    ax.set_xticks(range(1, 7))
    ax.set_ylim(0.3, 0.65)
        
    # 半包边框（只保留 bottom 和 left）
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.8)
    ax.spines['bottom'].set_linewidth(0.8)

    #ax.grid(True, alpha=0.25, linestyle="--")
    ax.legend(["Observed AUC", "Chance level"], frameon=False, fontsize=6.8, loc="lower right")
    plt.tight_layout()
    save_figure(fig, "fig1_decode_auc_by_reveal")
    plt.close(fig)
    print("✓ 已保存: fig1_decode_auc_by_reveal.pdf")

# =========================================================
# 7) concise summary
# =========================================================
summary = []

summary.append("=== Overall decode of bestdeck_changed_after_reveal from ACC ===")
summary.append(str(decode_df[decode_df["scope"] == "overall"]))
summary.append("")

summary.append("=== Reveal-wise decode ===")
summary.append(str(decode_df[decode_df["scope"] == "by_reveal"]))
summary.append("")

summary.append("=== Switch frequency by reveal and condition ===")
summary.append(str(freq_df))
summary.append("")

# peak AUC
tmp = decode_df[decode_df["scope"] == "by_reveal"].dropna(subset=["auc"]).copy()
if len(tmp) > 0:
    best = tmp.loc[tmp["auc"].idxmax()]
    summary.append(
        f"Peak reveal-wise decode: reveal {int(best['reveal_idx'])}, AUC={best['auc']:.4f}, p={best['perm_p']:.4f}"
    )

# update means
upd_sum = (
    df_update.groupby(["reveal_idx", "bestdeck_changed_after_reveal"])["acc_state_update"]
    .mean()
    .reset_index()
)
summary.append("")
summary.append("=== Mean ACC state update by reveal and switch ===")
summary.append(str(upd_sum))

with open(os.path.join(OUTPUT_DIR, "quick_summary.txt"), "w") as f:
    f.write("\n".join(summary))

print("Done.")
print("Saved to:", OUTPUT_DIR)


开始生成精炼可视化...
✓ 已保存: fig1_decode_auc_by_reveal.pdf
Done.
Saved to: /root/autodl-tmp/acc_bestdeck_change_main_analysis/


# logistic decoding

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score

warnings.filterwarnings("ignore")

# =========================================================
# paths
# =========================================================
INPUT_CSV = "/root/acc_normative_full_analysis/df_state_with_normative_metrics.csv"
OUTPUT_DIR = "/root/acc_reveal3_switch_statespace_centroid_loso/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================================================
# config
# =========================================================
SUBJ_COL = "Subject"
GAME_COL = "Game"
REVEAL_COL = "reveal_idx"
TARGET_COL = "bestdeck_changed_after_reveal"
ACC_COLS = ["acc_PC1", "acc_PC2", "acc_PC3"]

TARGET_REVEAL = 3
N_PERM = 5000
RANDOM_STATE = 42

# =========================================================
# load
# =========================================================
df = pd.read_csv(INPUT_CSV)
print("Loaded:", df.shape)

required_cols = [SUBJ_COL, GAME_COL, REVEAL_COL, TARGET_COL] + ACC_COLS
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

for c in [REVEAL_COL, TARGET_COL] + ACC_COLS:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=required_cols).copy()

if "game_complete_6" in df.columns:
    df = df[df["game_complete_6"] == True].copy()

df = df[df[TARGET_COL].isin([0, 1])].copy()
df[TARGET_COL] = df[TARGET_COL].astype(int)

df = df[df[REVEAL_COL] == TARGET_REVEAL].copy()
df = df.sort_values([SUBJ_COL, GAME_COL]).reset_index(drop=True)

print(f"Reveal {TARGET_REVEAL} rows:", df.shape[0])
print("Subjects:", df[SUBJ_COL].nunique())
print(df[TARGET_COL].value_counts().sort_index())

# =========================================================
# helpers
# =========================================================
def euclid(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    return float(np.sqrt(np.sum((a - b) ** 2)))

def safe_auc(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score, dtype=float)
    if len(np.unique(y_true)) < 2:
        return np.nan
    try:
        return roc_auc_score(y_true, y_score)
    except Exception:
        return np.nan

def bootstrap_mean_ci(x, n_boot=5000, alpha=0.05, random_state=42):
    x = np.asarray(pd.Series(x).dropna(), dtype=float)
    if len(x) == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(random_state)
    boots = []
    for _ in range(n_boot):
        samp = rng.choice(x, size=len(x), replace=True)
        boots.append(np.mean(samp))
    lo = np.percentile(boots, 100 * (alpha / 2))
    hi = np.percentile(boots, 100 * (1 - alpha / 2))
    return np.mean(x), lo, hi

def bootstrap_centroid_distance(subdf, n_boot=5000, random_state=42):
    """
    bootstrap overall centroid distance between switch and no-switch
    """
    rng = np.random.default_rng(random_state)
    g0 = subdf[subdf[TARGET_COL] == 0][ACC_COLS].to_numpy(dtype=float)
    g1 = subdf[subdf[TARGET_COL] == 1][ACC_COLS].to_numpy(dtype=float)

    if len(g0) == 0 or len(g1) == 0:
        return np.nan, np.nan, np.nan

    obs = euclid(g0.mean(axis=0), g1.mean(axis=0))

    boots = []
    for _ in range(n_boot):
        b0 = g0[rng.integers(0, len(g0), len(g0))]
        b1 = g1[rng.integers(0, len(g1), len(g1))]
        boots.append(euclid(b0.mean(axis=0), b1.mean(axis=0)))

    lo = np.percentile(boots, 2.5)
    hi = np.percentile(boots, 97.5)
    return obs, lo, hi

def permutation_centroid_distance(subdf, n_perm=2000, random_state=42):
    """
    permutation test for centroid distance
    """
    rng = np.random.default_rng(random_state)

    X = subdf[ACC_COLS].to_numpy(dtype=float)
    y = subdf[TARGET_COL].to_numpy(dtype=int)

    if len(np.unique(y)) < 2:
        return np.nan, np.nan

    c0 = X[y == 0].mean(axis=0)
    c1 = X[y == 1].mean(axis=0)
    obs = euclid(c0, c1)

    perm_scores = []
    for _ in range(n_perm):
        yp = rng.permutation(y)
        if len(np.unique(yp)) < 2:
            continue
        c0p = X[yp == 0].mean(axis=0)
        c1p = X[yp == 1].mean(axis=0)
        perm_scores.append(euclid(c0p, c1p))

    perm_scores = np.asarray(perm_scores, dtype=float)
    if len(perm_scores) == 0:
        return obs, np.nan

    p = (np.sum(perm_scores >= obs) + 1) / (len(perm_scores) + 1)
    return obs, p

def subject_level_centroid_distances(subdf):
    """
    each subject's within-subject centroid distance
    """
    rows = []
    for subj, ss in subdf.groupby(SUBJ_COL):
        ss = ss.copy()
        y = ss[TARGET_COL].to_numpy(dtype=int)
        if len(np.unique(y)) < 2:
            dist = np.nan
            c0 = [np.nan, np.nan, np.nan]
            c1 = [np.nan, np.nan, np.nan]
        else:
            X0 = ss[ss[TARGET_COL] == 0][ACC_COLS].to_numpy(dtype=float)
            X1 = ss[ss[TARGET_COL] == 1][ACC_COLS].to_numpy(dtype=float)
            c0 = X0.mean(axis=0)
            c1 = X1.mean(axis=0)
            dist = euclid(c0, c1)

        rows.append({
            "Subject": subj,
            "n_trials": len(ss),
            "n_switch0": int(np.sum(y == 0)),
            "n_switch1": int(np.sum(y == 1)),
            "centroid_distance": dist,
            "centroid0_PC1": c0[0],
            "centroid0_PC2": c0[1],
            "centroid0_PC3": c0[2],
            "centroid1_PC1": c1[0],
            "centroid1_PC2": c1[1],
            "centroid1_PC3": c1[2],
        })
    return pd.DataFrame(rows)

def loso_distance_to_centroid(subdf):
    """
    LOSO distance-to-centroid classification.
    Train centroids on N-1 subjects, test on held-out subject.
    Prediction score:
        d(no-switch centroid) - d(switch centroid)
    larger => more switch-like
    """
    subjects = sorted(subdf[SUBJ_COL].unique())
    fold_rows = []
    pooled_scores = []
    pooled_y = []
    pooled_pred = []

    for test_subj in subjects:
        train = subdf[subdf[SUBJ_COL] != test_subj].copy()
        test = subdf[subdf[SUBJ_COL] == test_subj].copy()

        ytr = train[TARGET_COL].to_numpy(dtype=int)
        yte = test[TARGET_COL].to_numpy(dtype=int)

        if len(np.unique(ytr)) < 2 or len(np.unique(yte)) < 2:
            fold_rows.append({
                "test_subject": test_subj,
                "auc": np.nan,
                "accuracy": np.nan,
                "balanced_accuracy": np.nan,
                "n_test": len(test),
                "n_test_switch0": int(np.sum(yte == 0)),
                "n_test_switch1": int(np.sum(yte == 1)),
                "train_centroid0_PC1": np.nan,
                "train_centroid0_PC2": np.nan,
                "train_centroid0_PC3": np.nan,
                "train_centroid1_PC1": np.nan,
                "train_centroid1_PC2": np.nan,
                "train_centroid1_PC3": np.nan,
                "train_centroid_distance": np.nan,
            })
            continue

        Xtr0 = train[train[TARGET_COL] == 0][ACC_COLS].to_numpy(dtype=float)
        Xtr1 = train[train[TARGET_COL] == 1][ACC_COLS].to_numpy(dtype=float)

        c0 = Xtr0.mean(axis=0)  # no-switch centroid
        c1 = Xtr1.mean(axis=0)  # switch centroid

        Xte = test[ACC_COLS].to_numpy(dtype=float)

        d0 = np.sqrt(np.sum((Xte - c0) ** 2, axis=1))
        d1 = np.sqrt(np.sum((Xte - c1) ** 2, axis=1))

        # score > 0 means closer to switch centroid
        score = d0 - d1
        pred = (score > 0).astype(int)

        auc = safe_auc(yte, score)
        acc = accuracy_score(yte, pred)
        bacc = balanced_accuracy_score(yte, pred)

        fold_rows.append({
            "test_subject": test_subj,
            "auc": auc,
            "accuracy": acc,
            "balanced_accuracy": bacc,
            "n_test": len(test),
            "n_test_switch0": int(np.sum(yte == 0)),
            "n_test_switch1": int(np.sum(yte == 1)),
            "train_centroid0_PC1": c0[0],
            "train_centroid0_PC2": c0[1],
            "train_centroid0_PC3": c0[2],
            "train_centroid1_PC1": c1[0],
            "train_centroid1_PC2": c1[1],
            "train_centroid1_PC3": c1[2],
            "train_centroid_distance": euclid(c0, c1),
        })

        pooled_scores.extend(score.tolist())
        pooled_y.extend(yte.tolist())
        pooled_pred.extend(pred.tolist())

    fold_df = pd.DataFrame(fold_rows)

    overall_auc = safe_auc(np.asarray(pooled_y), np.asarray(pooled_scores))
    overall_acc = accuracy_score(pooled_y, pooled_pred) if len(pooled_y) > 0 else np.nan
    overall_bacc = balanced_accuracy_score(pooled_y, pooled_pred) if len(pooled_y) > 0 else np.nan

    mean_subject_auc = fold_df["auc"].mean()
    mean_subject_bacc = fold_df["balanced_accuracy"].mean()

    return overall_auc, overall_acc, overall_bacc, mean_subject_auc, mean_subject_bacc, fold_df

def loso_distance_to_centroid_permutation(subdf, n_perm=2000, random_state=42):
    """
    permute labels within subject to preserve subject-specific base rates
    """
    rng = np.random.default_rng(random_state)
    obs_auc, obs_acc, obs_bacc, obs_mean_auc, obs_mean_bacc, fold_df = loso_distance_to_centroid(subdf)

    perm_auc = []
    perm_bacc = []
    perm_mean_auc = []
    perm_mean_bacc = []

    base = subdf.copy()

    for _ in range(n_perm):
        tmp = base.copy()
        tmp[TARGET_COL] = tmp.groupby(SUBJ_COL)[TARGET_COL].transform(
            lambda x: rng.permutation(x.values)
        )

        a, _, b, ma, mb, _ = loso_distance_to_centroid(tmp)
        if not np.isnan(a):
            perm_auc.append(a)
        if not np.isnan(b):
            perm_bacc.append(b)
        if not np.isnan(ma):
            perm_mean_auc.append(ma)
        if not np.isnan(mb):
            perm_mean_bacc.append(mb)

    def calc_p(obs, perm_vals):
        perm_vals = np.asarray(perm_vals, dtype=float)
        if len(perm_vals) == 0 or np.isnan(obs):
            return np.nan
        return (np.sum(perm_vals >= obs) + 1) / (len(perm_vals) + 1)

    out = {
        "obs_auc": obs_auc,
        "obs_acc": obs_acc,
        "obs_bacc": obs_bacc,
        "obs_mean_subject_auc": obs_mean_auc,
        "obs_mean_subject_bacc": obs_mean_bacc,
        "perm_p_auc": calc_p(obs_auc, perm_auc),
        "perm_p_bacc": calc_p(obs_bacc, perm_bacc),
        "perm_p_mean_subject_auc": calc_p(obs_mean_auc, perm_mean_auc),
        "perm_p_mean_subject_bacc": calc_p(obs_mean_bacc, perm_mean_bacc),
        "fold_df": fold_df
    }
    return out

# =========================================================
# main tables
# =========================================================
# overall centroids
overall_centroids = (
    df.groupby(TARGET_COL)[ACC_COLS]
      .mean()
      .reset_index()
      .rename(columns={TARGET_COL: "switch_label"})
)
overall_centroids.to_csv(os.path.join(OUTPUT_DIR, "reveal3_overall_centroids.csv"), index=False)

# subject-level centroid distances
subj_cent_df = subject_level_centroid_distances(df)
subj_cent_df.to_csv(os.path.join(OUTPUT_DIR, "reveal3_subject_centroid_distances.csv"), index=False)

# overall centroid separation
obs_dist, ci_lo, ci_hi = bootstrap_centroid_distance(df, n_boot=5000, random_state=RANDOM_STATE)
obs_dist_perm, p_cent = permutation_centroid_distance(df, n_perm=N_PERM, random_state=RANDOM_STATE)

centroid_summary = pd.DataFrame([{
    "reveal_idx": TARGET_REVEAL,
    "overall_centroid_distance": obs_dist,
    "bootstrap_ci_low": ci_lo,
    "bootstrap_ci_high": ci_hi,
    "perm_p_centroid_distance": p_cent,
    "n_trials": len(df),
    "n_switch0": int(np.sum(df[TARGET_COL] == 0)),
    "n_switch1": int(np.sum(df[TARGET_COL] == 1)),
    "n_subjects": int(df[SUBJ_COL].nunique()),
}])
centroid_summary.to_csv(os.path.join(OUTPUT_DIR, "reveal3_centroid_separation_summary.csv"), index=False)

# subject-level mean centroid separation
mean_subj_dist, subj_lo, subj_hi = bootstrap_mean_ci(
    subj_cent_df["centroid_distance"].dropna().values,
    n_boot=5000,
    random_state=RANDOM_STATE
)
subject_centroid_summary = pd.DataFrame([{
    "reveal_idx": TARGET_REVEAL,
    "mean_subject_centroid_distance": mean_subj_dist,
    "bootstrap_ci_low": subj_lo,
    "bootstrap_ci_high": subj_hi,
    "n_subjects_valid": int(np.sum(~subj_cent_df["centroid_distance"].isna()))
}])
subject_centroid_summary.to_csv(os.path.join(OUTPUT_DIR, "reveal3_subject_centroid_distance_summary.csv"), index=False)

# =========================================================
# LOSO distance-to-centroid classification
# =========================================================
loso_res = loso_distance_to_centroid_permutation(df, n_perm=N_PERM, random_state=RANDOM_STATE)
loso_fold_df = loso_res["fold_df"].copy()
loso_fold_df["reveal_idx"] = TARGET_REVEAL
loso_fold_df.to_csv(os.path.join(OUTPUT_DIR, "reveal3_loso_distance_to_centroid_fold_details.csv"), index=False)

loso_summary = pd.DataFrame([{
    "reveal_idx": TARGET_REVEAL,
    "loso_pooled_auc": loso_res["obs_auc"],
    "loso_pooled_accuracy": loso_res["obs_acc"],
    "loso_pooled_balanced_accuracy": loso_res["obs_bacc"],
    "loso_mean_subject_auc": loso_res["obs_mean_subject_auc"],
    "loso_mean_subject_balanced_accuracy": loso_res["obs_mean_subject_bacc"],
    "perm_p_auc": loso_res["perm_p_auc"],
    "perm_p_bacc": loso_res["perm_p_bacc"],
    "perm_p_mean_subject_auc": loso_res["perm_p_mean_subject_auc"],
    "perm_p_mean_subject_bacc": loso_res["perm_p_mean_subject_bacc"],
    "n_trials": len(df),
    "n_switch0": int(np.sum(df[TARGET_COL] == 0)),
    "n_switch1": int(np.sum(df[TARGET_COL] == 1)),
    "n_subjects": int(df[SUBJ_COL].nunique())
}])
loso_summary.to_csv(os.path.join(OUTPUT_DIR, "reveal3_loso_distance_to_centroid_summary.csv"), index=False)

# =========================================================
# plots
# =========================================================

# ---------- Plot 1: pooled state-space ----------
fig, ax = plt.subplots(figsize=(7.2, 6.0))

g0 = df[df[TARGET_COL] == 0].copy()
g1 = df[df[TARGET_COL] == 1].copy()

ax.scatter(
    g0["acc_PC1"], g0["acc_PC2"],
    s=26, alpha=0.35, label="no-switch", marker="o"
)
ax.scatter(
    g1["acc_PC1"], g1["acc_PC2"],
    s=38, alpha=0.55, label="switch", marker="^"
)

c0 = g0[ACC_COLS].mean().to_numpy(dtype=float)
c1 = g1[ACC_COLS].mean().to_numpy(dtype=float)

ax.scatter(c0[0], c0[1], s=220, marker="X", label="no-switch centroid")
ax.scatter(c1[0], c1[1], s=220, marker="P", label="switch centroid")

ax.plot([c0[0], c1[0]], [c0[1], c1[1]], linewidth=2)

ax.set_xlabel("ACC PC1")
ax.set_ylabel("ACC PC2")
ax.set_title(
    f"Reveal {TARGET_REVEAL}: ACC state-space (switch vs no-switch)\n"
    f"centroid distance = {obs_dist:.3f}, perm p = {p_cent:.3f}"
)
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "fig1_reveal3_statespace_switch_vs_noswitch.png"), dpi=250)
plt.close()

# ---------- Plot 2: subject centroids ----------
fig, ax = plt.subplots(figsize=(7.2, 6.0))

for _, r in subj_cent_df.dropna(subset=["centroid_distance"]).iterrows():
    x0, y0 = r["centroid0_PC1"], r["centroid0_PC2"]
    x1, y1 = r["centroid1_PC1"], r["centroid1_PC2"]

    ax.plot([x0, x1], [y0, y1], alpha=0.7, linewidth=1.5)
    ax.scatter(x0, y0, s=45, marker="o")
    ax.scatter(x1, y1, s=55, marker="^")
    ax.text(x0, y0, str(int(r["Subject"])), fontsize=8, alpha=0.8)
    ax.text(x1, y1, str(int(r["Subject"])), fontsize=8, alpha=0.8)

ax.scatter(c0[0], c0[1], s=250, marker="X", label="overall no-switch centroid")
ax.scatter(c1[0], c1[1], s=250, marker="P", label="overall switch centroid")

ax.set_xlabel("ACC PC1")
ax.set_ylabel("ACC PC2")
ax.set_title(
    f"Reveal {TARGET_REVEAL}: subject-level centroids\n"
    f"mean subject centroid distance = {mean_subj_dist:.3f}"
)
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "fig2_reveal3_subject_centroid_pairs.png"), dpi=250)
plt.close()

# ---------- Plot 3: subject centroid distances ----------
fig, ax = plt.subplots(figsize=(7.0, 5.4))

tmp = subj_cent_df.sort_values("centroid_distance").copy()
ax.bar(tmp[SUBJ_COL].astype(str), tmp["centroid_distance"])
ax.axhline(mean_subj_dist, linestyle="--", linewidth=1.5)
ax.set_xlabel("Subject")
ax.set_ylabel("Within-subject centroid distance")
ax.set_title(
    f"Reveal {TARGET_REVEAL}: centroid separation by subject\n"
    f"mean = {mean_subj_dist:.3f}, 95% CI [{subj_lo:.3f}, {subj_hi:.3f}]"
)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "fig3_reveal3_subject_centroid_distance_bar.png"), dpi=250)
plt.close()

# ---------- Plot 4: LOSO distance-to-centroid AUC by subject ----------
fig, ax = plt.subplots(figsize=(7.0, 5.4))

tmp = loso_fold_df.sort_values("auc").copy()
ax.bar(tmp["test_subject"].astype(str), tmp["auc"])
ax.axhline(0.5, linestyle="--", linewidth=1)
ax.axhline(loso_res["obs_mean_subject_auc"], linestyle="--", linewidth=1.5)
ax.set_xlabel("Held-out subject")
ax.set_ylabel("LOSO distance-to-centroid AUC")
ax.set_title(
    f"Reveal {TARGET_REVEAL}: LOSO distance-to-centroid decoding\n"
    f"pooled AUC = {loso_res['obs_auc']:.3f}, perm p = {loso_res['perm_p_auc']:.3f}"
)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "fig4_reveal3_loso_distance_to_centroid_auc.png"), dpi=250)
plt.close()

# ---------- Plot 5: decision score distributions ----------
# score = d(no-switch centroid) - d(switch centroid); higher => more switch-like
score_rows = []
subjects = sorted(df[SUBJ_COL].unique())

for test_subj in subjects:
    train = df[df[SUBJ_COL] != test_subj].copy()
    test = df[df[SUBJ_COL] == test_subj].copy()

    ytr = train[TARGET_COL].to_numpy(dtype=int)
    yte = test[TARGET_COL].to_numpy(dtype=int)

    if len(np.unique(ytr)) < 2 or len(np.unique(yte)) < 2:
        continue

    c0_train = train[train[TARGET_COL] == 0][ACC_COLS].mean().to_numpy(dtype=float)
    c1_train = train[train[TARGET_COL] == 1][ACC_COLS].mean().to_numpy(dtype=float)

    Xte = test[ACC_COLS].to_numpy(dtype=float)
    d0 = np.sqrt(np.sum((Xte - c0_train) ** 2, axis=1))
    d1 = np.sqrt(np.sum((Xte - c1_train) ** 2, axis=1))
    score = d0 - d1

    out = test[[SUBJ_COL, GAME_COL, TARGET_COL]].copy()
    out["decision_score"] = score
    score_rows.append(out)

if len(score_rows) > 0:
    score_df = pd.concat(score_rows, axis=0, ignore_index=True)
    score_df.to_csv(os.path.join(OUTPUT_DIR, "reveal3_loso_trialwise_decision_scores.csv"), index=False)

    fig, ax = plt.subplots(figsize=(7.0, 5.4))
    s0 = score_df[score_df[TARGET_COL] == 0]["decision_score"].dropna().values
    s1 = score_df[score_df[TARGET_COL] == 1]["decision_score"].dropna().values

    ax.hist(s0, bins=30, alpha=0.5, density=True, label="no-switch")
    ax.hist(s1, bins=30, alpha=0.5, density=True, label="switch")
    ax.axvline(0, linestyle="--", linewidth=1)
    ax.set_xlabel("Decision score: d(no-switch centroid) - d(switch centroid)")
    ax.set_ylabel("Density")
    ax.set_title(f"Reveal {TARGET_REVEAL}: LOSO decision score distributions")
    ax.legend(frameon=False)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "fig5_reveal3_loso_decision_score_hist.png"), dpi=250)
    plt.close()

# =========================================================
# text summary
# =========================================================
summary_lines = []
summary_lines.append("=== Reveal 3 ACC switch vs no-switch state-space summary ===")
summary_lines.append(f"N trials: {len(df)}")
summary_lines.append(f"N subjects: {df[SUBJ_COL].nunique()}")
summary_lines.append(f"N no-switch: {int(np.sum(df[TARGET_COL] == 0))}")
summary_lines.append(f"N switch: {int(np.sum(df[TARGET_COL] == 1))}")
summary_lines.append("")
summary_lines.append("=== Overall centroid separation ===")
summary_lines.append(f"Centroid distance: {obs_dist:.6f}")
summary_lines.append(f"Bootstrap 95% CI: [{ci_lo:.6f}, {ci_hi:.6f}]")
summary_lines.append(f"Permutation p: {p_cent:.6f}")
summary_lines.append("")
summary_lines.append("=== Subject-level centroid separation ===")
summary_lines.append(f"Mean subject centroid distance: {mean_subj_dist:.6f}")
summary_lines.append(f"Bootstrap 95% CI: [{subj_lo:.6f}, {subj_hi:.6f}]")
summary_lines.append("")
summary_lines.append("=== LOSO distance-to-centroid decoding ===")
summary_lines.append(f"Pooled AUC: {loso_res['obs_auc']:.6f}")
summary_lines.append(f"Pooled accuracy: {loso_res['obs_acc']:.6f}")
summary_lines.append(f"Pooled balanced accuracy: {loso_res['obs_bacc']:.6f}")
summary_lines.append(f"Mean subject AUC: {loso_res['obs_mean_subject_auc']:.6f}")
summary_lines.append(f"Mean subject balanced accuracy: {loso_res['obs_mean_subject_bacc']:.6f}")
summary_lines.append(f"Permutation p (AUC): {loso_res['perm_p_auc']:.6f}")
summary_lines.append(f"Permutation p (balanced accuracy): {loso_res['perm_p_bacc']:.6f}")
summary_lines.append(f"Permutation p (mean subject AUC): {loso_res['perm_p_mean_subject_auc']:.6f}")
summary_lines.append(f"Permutation p (mean subject balanced accuracy): {loso_res['perm_p_mean_subject_bacc']:.6f}")
summary_lines.append("")

with open(os.path.join(OUTPUT_DIR, "reveal3_summary.txt"), "w") as f:
    f.write("\n".join(summary_lines))

print("\nDone. Results saved to:", OUTPUT_DIR)
print("\n".join(summary_lines))

Loaded: (5166, 264)
Reveal 3 rows: 861
Subjects: 10
bestdeck_changed_after_reveal
0    711
1    150
Name: count, dtype: int64

Done. Results saved to: /root/autodl-tmp/acc_reveal3_switch_statespace_centroid_loso/
=== Reveal 3 ACC switch vs no-switch state-space summary ===
N trials: 861
N subjects: 10
N no-switch: 711
N switch: 150

=== Overall centroid separation ===
Centroid distance: 0.548244
Bootstrap 95% CI: [0.227707, 1.120091]
Permutation p: 0.161919

=== Subject-level centroid separation ===
Mean subject centroid distance: 1.478001
Bootstrap 95% CI: [0.969462, 2.002690]

=== LOSO distance-to-centroid decoding ===
Pooled AUC: 0.555827
Pooled accuracy: 0.535424
Pooled balanced accuracy: 0.534599
Mean subject AUC: 0.561366
Mean subject balanced accuracy: 0.526737
Permutation p (AUC): 0.010495
Permutation p (balanced accuracy): 0.082459
Permutation p (mean subject AUC): 0.026987
Permutation p (mean subject balanced accuracy): 0.175912

